# Brain Tumor Segmentation - Complete Implementation
## Res-UNet 3D with Multimodal Fusion

This notebook implements a comprehensive brain tumor segmentation pipeline following the research gaps identified in the project report.

**Phases:**
- Phase 1: Setup and Configuration(Cells 1 - 3)(Shruthi)
- Phase 2: Data Loading and Preprocessing(Cells 4 - 8)(Shruthi)
- Phase 3: Model Architecture(Cells 9 - 12)(Hrishikesh(9 - 11 cells) and Shruthi(12th cell))
- Phase 4: Loss Functions and Metrics(Cells 13 - 15)(Hrishikesh(13 - 14 cells) and Nihal(15th cell))
- Phase 5: Training Pipeline(Cells 15 - 20)(Hrishikesh)
- Phase 6: Testing and Final Evaluation(Cells 20 - 24)(Hrishikesh(20 - 22 cells) and Nihal(22 - 24 cells))
- Phase 7: Ablation Studies(Cells 25 - 26)(Nihal)
- Phase 8: Export and Documentation(Cell 27)(Nihal)
- Phase 9: Automated Execution Pipelines(Cells 28 - 29)(Nihal)

---

## 👥 Team Contributions

This notebook is a **collaborative effort** with contributions from three team members, each handling major technical components of the pipeline.

---

### 🔷 **Shruthi**

**Phase 1: Setup and Configuration** (`Cells 1-3`)
- Imports and dependencies setup
- Configuration class design
- Dataset path setup and environment detection

**Phase 2: Data Loading and Preprocessing** (`Cells 4-8`)
- NIfTI/MHA file loading utilities (BraTS 2021/2015 support)
- Preprocessing functions (normalization, padding, brain mask cropping)
- Tumor-centric and class-aware patch sampling
- Dataset class implementation with fusion support
- DataLoader setup and configuration

**Phase 3: Model Architecture** (`Cell 12`)
- Learnable fusion model implementation

---

### 🔷 **Hrishikesh**

**Phase 3: Model Architecture** (`Cells 9-11`)
- ResUNet3D backbone architecture
- Residual blocks and skip connections
- Model creation utilities

**Phase 4: Loss Functions and Metrics** (`Cells 13-14`)
- Dice loss, Focal loss, and combined loss implementations
- Loss smoothing utilities (EMA)

**Phase 5: Training Pipeline** (`Cells 15-20`)
- Training setup (optimizer, scheduler, scaler)
- Training and validation epoch functions
- Full training loop with checkpointing
- Model loading utilities
- Training visualization and plotting

**Phase 6: Testing and Final Evaluation** (`Cells 20-24`)
- Test set evaluation pipeline
- Comprehensive metrics computation
- Prediction visualization

---

### 🔷 **Nihal**

**Phase 4: Loss Functions and Metrics** (`Cell 15`)
- Robust evaluation function with deterministic multi-batch evaluation

**Phase 7: Ablation Studies** (`Cells 25-27`)
- Fusion ablation study (mean, static, learnable fusion)
- Loss function ablation study (Dice, Focal, Combined, CE)
- Architecture ablation study (PlainUNet, SimpleSwin, ResUNet variants)

**Phase 8: Export and Documentation** (`Cell 28`)
- Metrics export (CSV, JSON)
- Training history export
- Configuration export
- Final results compilation

**Phase 9: Automated Execution Pipelines** (`Cells 29-30`)
- Training phase orchestration (data preprocessing, training, plotting)
- Evaluation phase orchestration (model loading, evaluation, ablations, export)

---

## Phase 1: Setup and Configuration(Shruthi)

### Cell 1: Imports and Dependencies(Shruthi)


In [ ]:
# Cell 1: Imports and Dependencies
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os
import glob
import random
import math
import copy
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Union
from dataclasses import dataclass
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from tqdm import tqdm

# Medical image readers
try:
    import SimpleITK as sitk
except ImportError:
    sitk = None
    print("Warning: SimpleITK not available. Install with: pip install SimpleITK")

# Set device (CUDA if available)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using CUDA device: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS device")
else:
    device = torch.device("cpu")
    print("Using CPU device")

# Enable reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"Random seed set to: {RANDOM_SEED}")
print(f"PyTorch version: {torch.__version__}")


### Cell 2: Configuration Class(Shruthi)


In [ ]:
# Cell 2: Configuration Class
@dataclass
class TrainConfig:
    """Configuration class for training parameters tuned for Kaggle T4 runtime"""
    # Data paths
    data_root: str = ""
    output_dir: str = "./outputs"
    checkpoint_dir: str = "./checkpoints"
    
    # Hyperparameters (balanced for ~6-7 hr Kaggle T4 runs)
    batch_size: int = 4 # Gradient accumulation simulates larger batches
    learning_rate: float = 3e-4
    epochs: int = 25
    patch_size: Tuple[int, int, int] = (98, 98, 98)
    num_workers: int = 2
    gradient_accumulation_steps: int = 4
    
    # Data split / sampling
    train_split_ratio: float = 0.8
    val_split_ratio: float = 0.2
    train_patches_per_case: int = 12
    val_patches_per_case: int = 4
    tumor_centric_ratio: float = 0.8
    class_aware_sampling: bool = True
    minority_class_ratio: float = 0.3
    
    # Model config
    base_filters: int = 16
    depth: int = 4
    num_classes: int = 5  # background, necrosis, edema, non-enhancing, enhancing
    use_instance_norm: bool = True
    use_learnable_fusion: bool = True
    
    # Dropout config
    dropout_rate_deep: float = 0.25
    dropout_rate_shallow: float = 0.1
    dropout_enabled: bool = True
    
    # Training
    optimizer_type: str = 'AdamW'
    scheduler_type: str = 'CosineAnnealingLR'
    gradient_clip: float = 0.5
    
    # Loss config
    dice_weight: float = 0.4
    focal_weight: float = 0.6
    focal_alpha: List[float] = None  # [bg, necrosis, edema, non-enh, enh]
    focal_gamma: float = 2.0
    
    # Evaluation
    eval_batches: int = 12
    use_center_crop_eval: bool = True
    
    def __post_init__(self):
        """Set defaults and ensure filesystem targets exist"""
        if self.focal_alpha is None:
            self.focal_alpha = [0.1, 2.0, 2.0, 3.0, 3.0]  # [bg, necrosis, edema, non-enh, enh]
        
        # Enforce split ratios (validation is always the leftover portion)
        self.train_split_ratio = min(max(self.train_split_ratio, 0.7), 0.95)
        self.val_split_ratio = round(1.0 - self.train_split_ratio, 2)
        
        # Create output directories
        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(self.checkpoint_dir, exist_ok=True)

# Initialize default config
config = TrainConfig()
print("Configuration initialized (T4-ready):")
print(f"  Batch size: {config.batch_size} (grad accum: {config.gradient_accumulation_steps})")
print(f"  Learning rate: {config.learning_rate}")
print(f"  Patch size: {config.patch_size}")
print(f"  Epochs: {config.epochs}")
print(f"  Train/Val split: {config.train_split_ratio*100:.0f}%/{config.val_split_ratio*100:.0f}%")
print(f"  Train patches per case: {config.train_patches_per_case}")
print(f"  Validation patches per case: {config.val_patches_per_case}")
print(f"  Dropout (deep/shallow): {config.dropout_rate_deep}/{config.dropout_rate_shallow}")
print(f"  Tumor-centric sampling: {config.tumor_centric_ratio*100}%")


### Cell 3: Dataset Path Setup(Shruthi)


In [ ]:
# Cell 3: Dataset Path Setup and Extraction (BraTS 2021)
def detect_environment():
    """Detect if running on Kaggle or local environment"""
    if os.path.exists("/kaggle/input"):
        return "kaggle"
    else:
        return "local"

def extract_brats2021_kaggle():
    """Extract BraTS 2021 TAR file in Kaggle environment"""
    env = detect_environment()
    if env != "kaggle":
        return None
    
    raw_path = "/kaggle/input/brats-2021-task1"
    work_dir = "/kaggle/working/brats2021"
    tar_file = os.path.join(raw_path, "BraTS2021_Training_Data.tar")
    
    # Check if already extracted
    if os.path.exists(work_dir) and len([d for d in os.listdir(work_dir) if d.startswith("BraTS2021")]) > 0:
        print(f"BraTS 2021 data already extracted at: {work_dir}")
        return work_dir
    
    # Extract if TAR file exists
    if os.path.exists(tar_file):
        print(f"Extracting BraTS 2021 dataset from {tar_file}...")
        os.makedirs(work_dir, exist_ok=True)
        import subprocess
        result = subprocess.run(
            ["tar", "-xf", tar_file, "-C", work_dir],
            capture_output=True,
            text=True
        )
        if result.returncode == 0:
            print(f"Extraction complete. Data available at: {work_dir}")
            return work_dir
        else:
            print(f"Extraction failed: {result.stderr}")
            return None
    else:
        print(f"TAR file not found at: {tar_file}")
        return None

def setup_data_paths(config: TrainConfig):
    """Setup data paths based on environment (BraTS 2021)"""
    env = detect_environment()
    
    # First, try to extract in Kaggle if needed
    extracted_path = extract_brats2021_kaggle()
    
    if env == "kaggle":
        # Kaggle dataset structure (BraTS 2021)
        possible_paths = [
            "/kaggle/working/brats2021",  # Extracted path
            "/kaggle/input/brats-2021-task1",  # Raw input path
            "/kaggle/input/brats2015/BRATS2015/training",  # Fallback to 2015
            "/kaggle/input/brats-2015/BRATS2015/training",  # Fallback to 2015
        ]
    else:
        # Local dataset structure
        possible_paths = [
            "./brats2021",  # Local extracted path
            "../brats2021",
            "Reviewed Literature/Implementation/BRATS2015/training",  # Fallback to 2015
            "./BRATS2015/training",  # Fallback to 2015
            "../BRATS2015/training",  # Fallback to 2015
        ]
    
    for path in possible_paths:
        if os.path.exists(path):
            # For BraTS 2021, check if it contains subject folders
            if any(d.startswith("BraTS2021") for d in os.listdir(path) if os.path.isdir(os.path.join(path, d))):
                config.data_root = path
                print(f"Found BraTS 2021 dataset at: {path}")
                return path
            # For BraTS 2015 fallback, check for HGG/LGG structure
            elif os.path.exists(os.path.join(path, "HGG")) or os.path.exists(os.path.join(path, "LGG")):
                config.data_root = path
                print(f"Found BraTS 2015 dataset at: {path}")
                return path
    
    print(f"Warning: Dataset not found in any of the expected paths:")
    for path in possible_paths:
        print(f"  - {path}")
    print("Please set config.data_root manually")
    return None

# Setup paths
data_root = setup_data_paths(config)
if data_root:
    config.data_root = data_root

def find_cases(data_root: str) -> List[str]:
    """Find all case directories in the dataset (supports both BraTS 2015 and 2021)"""
    if not data_root or not os.path.exists(data_root):
        return []
    
    cases = []
    
    # Check if BraTS 2021 structure (subject folders directly in data_root)
    subject_dirs = [d for d in os.listdir(data_root) 
                   if os.path.isdir(os.path.join(data_root, d)) and d.startswith("BraTS2021")]
    if subject_dirs:
        # BraTS 2021 structure: data_root/BraTS2021_XXXXX/
        for subject_dir in subject_dirs:
            subject_path = os.path.join(data_root, subject_dir)
            # Verify it has required modalities
            required_mods = ["flair", "t1", "t1ce", "t2", "seg"]
            has_all = all(os.path.exists(os.path.join(subject_path, f"{subject_dir}_{mod}.nii.gz")) 
                         for mod in required_mods)
            if has_all:
                cases.append(subject_path)
        print(f"Found {len(cases)} BraTS 2021 cases in dataset")
        return sorted(cases)
    
    # Fallback to BraTS 2015 structure: training/HGG/ and training/LGG/
    for grade_dir in ["HGG", "LGG"]:
        grade_path = os.path.join(data_root, grade_dir)
        if os.path.exists(grade_path):
            case_dirs = [d for d in os.listdir(grade_path) 
                        if os.path.isdir(os.path.join(grade_path, d))]
            cases.extend([os.path.join(grade_path, d) for d in case_dirs])
    
    if cases:
        print(f"Found {len(cases)} BraTS 2015 cases in dataset")
    else:
        print("No cases found in dataset")
    return sorted(cases)

# Find available cases (entire training split)
train_cases, val_cases, sampler_cases = [], [], []
if config.data_root:
    all_cases = find_cases(config.data_root)
    all_cases = all_cases[:1000]
    total_cases = len(all_cases)
    print(f"Total cases available: {total_cases}")
    if total_cases > 0:
        preview_count = min(3, total_cases)
        preview_cases = [os.path.basename(c) for c in all_cases[:preview_count]]
        print(f"Preview ({preview_count}): {preview_cases}")
        print("Using entire BraTS training folder; testing split is skipped (no labels).")
        val_ratio = config.val_split_ratio
        train_cases, val_cases = train_test_split(
            all_cases,
            test_size=val_ratio,
            random_state=RANDOM_SEED,
            shuffle=True
        )
        sampler_cases = train_cases + val_cases
        print(f"Train cases: {len(train_cases)} ({config.train_split_ratio*100:.0f}%)")
        print(f"Validation cases: {len(val_cases)} ({val_ratio*100:.0f}%)")
    else:
        print("No cases discovered under training directory.")
else:
    all_cases = []
    print("No dataset found. Please configure data_root manually.")


## Phase 2: Data Loading and Preprocessing(Shruthi)

### Cell 4: NIfTI/MHA File Loading Utilities (BraTS 2021/2015)(Shruthi)


In [ ]:
# Cell 4: NIfTI/MHA File Loading Utilities (BraTS 2021/2015)
def load_nifti_sitk(path: str):
    """
    Loads NIfTI using SimpleITK (safe for spacing, orientation, etc.)
    
    Returns:
        volume: np.ndarray (z,y,x)
        spacing: tuple (x_spacing, y_spacing, z_spacing)
    """
    if sitk is None:
        raise ImportError("SimpleITK is required. Install with: pip install SimpleITK")
    
    img = sitk.ReadImage(path)
    vol = sitk.GetArrayFromImage(img)  # converts to numpy, axis order = (z, y, x)
    spacing = img.GetSpacing()         # spacing = (x, y, z) — note ordering
    return vol.astype(np.float32), spacing

def load_case(case_path: str) -> Dict[str, np.ndarray]:
    """
    Load all 4 modalities + label for a case (supports both BraTS 2015 MHA and BraTS 2021 NIfTI)
    
    Args:
        case_path: Path to case directory
        
    Returns:
        Dictionary with keys ['t1', 't1ce', 't2', 'flair', 'label']
        Each value is a numpy array with shape (D, H, W)
        Also includes 'cropped_shape' and 'crop_slices' metadata if cropping was applied
    """
    if sitk is None:
        raise ImportError("SimpleITK is required. Install with: pip install SimpleITK")
    
    modalities = {
        't1': None,
        't1ce': None,
        't2': None,
        'flair': None,
        'label': None
    }
    
    case_id = os.path.basename(case_path)
    
    # Check if BraTS 2021 format (NIfTI files with naming: {case_id}_{modality}.nii.gz)
    nifti_files = glob.glob(os.path.join(case_path, "*.nii.gz"))
    if nifti_files:
        # BraTS 2021 format
        modality_map = {
            't1': f"{case_id}_t1.nii.gz",
            't1ce': f"{case_id}_t1ce.nii.gz",
            't2': f"{case_id}_t2.nii.gz",
            'flair': f"{case_id}_flair.nii.gz",
            'label': f"{case_id}_seg.nii.gz"
        }
        
        for mod_key, filename in modality_map.items():
            filepath = os.path.join(case_path, filename)
            if os.path.exists(filepath):
                vol, spacing = load_nifti_sitk(filepath)
                modalities[mod_key] = vol
            else:
                print(f"Warning: Missing {mod_key} file: {filename}")
    else:
        # BraTS 2015 format (MHA files)
        mha_files = glob.glob(os.path.join(case_path, "*.mha"))
        
        for mha_file in mha_files:
            filename = os.path.basename(mha_file).upper()
            
            # Identify modality from filename
            if 'T1C' in filename or 'T1CE' in filename or 'T1C.' in filename:
                reader = sitk.ImageFileReader()
                reader.SetFileName(mha_file)
                img = reader.Execute()
                modalities['t1ce'] = sitk.GetArrayFromImage(img).astype(np.float32)
            elif 'T1.' in filename and 'T1C' not in filename:
                reader = sitk.ImageFileReader()
                reader.SetFileName(mha_file)
                img = reader.Execute()
                modalities['t1'] = sitk.GetArrayFromImage(img).astype(np.float32)
            elif 'T2.' in filename:
                reader = sitk.ImageFileReader()
                reader.SetFileName(mha_file)
                img = reader.Execute()
                modalities['t2'] = sitk.GetArrayFromImage(img).astype(np.float32)
            elif 'FLAIR' in filename or 'FL' in filename:
                reader = sitk.ImageFileReader()
                reader.SetFileName(mha_file)
                img = reader.Execute()
                modalities['flair'] = sitk.GetArrayFromImage(img).astype(np.float32)
            elif 'OT' in filename or 'SEG' in filename or 'LABEL' in filename:
                reader = sitk.ImageFileReader()
                reader.SetFileName(mha_file)
                img = reader.Execute()
                modalities['label'] = sitk.GetArrayFromImage(img).astype(np.float32)
    
    # Apply mandatory brain mask cropping if label is available
    if modalities['label'] is not None:
        # Compute bounding box from non-zero label voxels
        nonzero = np.where(modalities['label'] != 0)
        if len(nonzero[0]) > 0:
            z1, z2 = nonzero[0].min(), nonzero[0].max()
            y1, y2 = nonzero[1].min(), nonzero[1].max()
            x1, x2 = nonzero[2].min(), nonzero[2].max()
            
            # Crop all modalities and label
            for mod_key in ['t1', 't1ce', 't2', 'flair', 'label']:
                if modalities[mod_key] is not None:
                    modalities[mod_key] = modalities[mod_key][z1:z2+1, y1:y2+1, x1:x2+1]
            
            # Store cropping metadata
            modalities['cropped_shape'] = (z2-z1+1, y2-y1+1, x2-x1+1)
            modalities['crop_slices'] = (slice(z1, z2+1), slice(y1, y2+1), slice(x1, x2+1))
        else:
            # No non-zero labels, store original shape
            if modalities['label'] is not None:
                modalities['cropped_shape'] = modalities['label'].shape
                modalities['crop_slices'] = (slice(None), slice(None), slice(None))
    else:
        # No label available, warn but continue
        print(f"Warning: No label found for case {case_id}. Cropping skipped.")
        modalities['cropped_shape'] = None
        modalities['crop_slices'] = None
    
    # Check if all modalities loaded
    missing = [k for k, v in modalities.items() if v is None and k not in ['cropped_shape', 'crop_slices']]
    if missing:
        print(f"Warning: Missing modalities for case {case_id}: {missing}")
    
    return modalities

# Alias for backward compatibility
load_mha_case = load_case

# Test loading (if cases available)
if len(all_cases) > 0:
    test_case = all_cases[0]
    print(f"Testing load for case: {os.path.basename(test_case)}")
    try:
        test_data = load_case(test_case)
        for mod, data in test_data.items():
            if mod not in ['cropped_shape', 'crop_slices'] and data is not None:
                print(f"  {mod}: shape={data.shape}, dtype={data.dtype}, range=[{data.min():.2f}, {data.max():.2f}]")
        if 'cropped_shape' in test_data:
            print(f"  cropped_shape: {test_data['cropped_shape']}")
    except Exception as e:
        print(f"Error loading test case: {e}")


### Cell 5: Preprocessing Functions(Shruthi)


In [ ]:
# Cell 5: Preprocessing Functions
def zscore_norm(img: np.ndarray, mask: Optional[np.ndarray] = None) -> np.ndarray:
    """
    Per-modality z-score normalization on non-zero voxels
    
    Args:
        img: Input image array (D, H, W)
        mask: Optional mask to define valid voxels (if None, uses non-zero voxels)
        
    Returns:
        Normalized image array
    """
    img = img.astype(np.float32)
    
    if mask is None:
        mask = img > 0
    
    if mask.sum() == 0:
        return img
    
    mean = img[mask].mean()
    std = img[mask].std()
    
    if std > 0:
        img = (img - mean) / std
    else:
        img = img - mean
    
    return img

def pad_to_min_size(vol: np.ndarray, min_size: Tuple[int, int, int]) -> np.ndarray:
    """
    Pad volume to minimum size
    
    Args:
        vol: Volume array (D, H, W)
        min_size: Minimum size (d, h, w)
        
    Returns:
        Padded volume
    """
    d, h, w = vol.shape
    min_d, min_h, min_w = min_size
    
    pad_d = max(0, min_d - d)
    pad_h = max(0, min_h - h)
    pad_w = max(0, min_w - w)
    
    if pad_d > 0 or pad_h > 0 or pad_w > 0:
        vol = np.pad(vol, ((0, pad_d), (0, pad_h), (0, pad_w)), mode='constant', constant_values=0)
    
    return vol

def get_tumor_centers(label: np.ndarray) -> List[Tuple[int, int, int]]:
    """
    Extract tumor center coordinates for tumor-centric sampling
    
    Args:
        label: Label array (D, H, W) with tumor classes (1-4)
        
    Returns:
        List of (z, y, x) coordinates of tumor centers
    """
    tumor_mask = (label > 0).astype(np.uint8)
    
    if tumor_mask.sum() == 0:
        return []
    
    # Find connected components
    from scipy.ndimage import label, center_of_mass
    labeled_mask, num_features = label(tumor_mask)
    
    centers = []
    for i in range(1, num_features + 1):
        component_mask = (labeled_mask == i)
        if component_mask.sum() > 0:
            center = center_of_mass(component_mask)
            centers.append((int(center[0]), int(center[1]), int(center[2])))
    
    return centers

def get_class_voxels(label: np.ndarray, class_id: int) -> List[Tuple[int, int, int]]:
    """
    Extract voxel coordinates for specific classes (for class-aware sampling)
    
    Args:
        label: Label array (D, H, W)
        class_id: Class ID to extract (1=necrosis, 2=edema, 3=non-enhancing, 4=enhancing)
        
    Returns:
        List of (z, y, x) coordinates
    """
    coords = np.where(label == class_id)
    return list(zip(coords[0], coords[1], coords[2]))

# Test preprocessing functions
if len(all_cases) > 0:
    test_case = all_cases[0]
    try:
        test_data = load_mha_case(test_case)
        if test_data['t1'] is not None:
            print("Testing preprocessing functions:")
            # Test z-score normalization
            t1_norm = zscore_norm(test_data['t1'])
            print(f"  Z-score norm - mean: {t1_norm[t1_norm>0].mean():.4f}, std: {t1_norm[t1_norm>0].std():.4f}")
            
            # Test tumor centers
            if test_data['label'] is not None:
                centers = get_tumor_centers(test_data['label'])
                print(f"  Found {len(centers)} tumor center(s)")
                
                # Test class voxels
                for class_id, name in [(1, "necrosis"), (2, "edema"), (3, "non-enhancing"), (4, "enhancing")]:
                    voxels = get_class_voxels(test_data['label'], class_id)
                    if len(voxels) > 0:
                        print(f"  {name} (class {class_id}): {len(voxels)} voxels")
    except Exception as e:
        print(f"Error testing preprocessing: {e}")


# Cell 6: Patch Extraction and Enhanced Sampling(Shruthi)

In [ ]:
# Cell 6: Patch Extraction and Enhanced Sampling
def extract_patch(vol: np.ndarray, center: Tuple[int, int, int], patch_size: Tuple[int, int, int]) -> np.ndarray:
    """
    Extract 3D patch at center
    
    Args:
        vol: Volume array (D, H, W)
        center: Center coordinates (z, y, x)
        patch_size: Patch size (d, h, w)
        
    Returns:
        Extracted patch (d, h, w)
    """
    d, h, w = vol.shape
    pd, ph, pw = patch_size
    cz, cy, cx = center
    
    # Calculate patch boundaries
    z0 = max(0, cz - pd // 2)
    z1 = min(d, z0 + pd)
    y0 = max(0, cy - ph // 2)
    y1 = min(h, y0 + ph)
    x0 = max(0, cx - pw // 2)
    x1 = min(w, x0 + pw)
    
    # Extract patch
    patch = vol[z0:z1, y0:y1, x0:x1]
    
    # Pad if necessary
    if patch.shape != patch_size:
        pad_z = max(0, pd - patch.shape[0])
        pad_y = max(0, ph - patch.shape[1])
        pad_x = max(0, pw - patch.shape[2])
        patch = np.pad(patch, ((0, pad_z), (0, pad_y), (0, pad_x)), mode='constant', constant_values=0)
    
    return patch

class TumorPatchSampler:
    """
    Enhanced class for tumor-centric and class-aware sampling
    """
    def __init__(self, cases: List[str], patch_size: Tuple[int, int, int], 
                 tumor_ratio: float = 0.8, minority_class_ratio: float = 0.3):
        """
        Args:
            cases: List of case paths
            patch_size: Patch size (d, h, w)
            tumor_ratio: Ratio of patches to sample from tumor centers (default 0.8)
            minority_class_ratio: Ratio of patches to contain minority classes (default 0.3)
        """
        self.cases = cases
        self.patch_size = patch_size
        self.tumor_ratio = tumor_ratio
        self.minority_class_ratio = minority_class_ratio
        
        # Pre-compute tumor centers and class voxels for each case
        self.case_info = {}
        self._precompute_case_info()
    
    def _precompute_case_info(self):
        """Pre-compute tumor centers and class voxels for faster sampling"""
        print("Pre-computing case information...")
        for case_path in tqdm(self.cases, desc="Loading case info"):
            try:
                data = load_mha_case(case_path)
                if data['label'] is not None:
                    centers = get_tumor_centers(data['label'])
                    minority_voxels = {
                        1: get_class_voxels(data['label'], 1),  # necrosis
                        3: get_class_voxels(data['label'], 3),   # non-enhancing
                    }
                    self.case_info[case_path] = {
                        'centers': centers,
                        'minority_voxels': minority_voxels,
                        'has_minority': len(minority_voxels[1]) > 0 or len(minority_voxels[3]) > 0
                    }
                else:
                    self.case_info[case_path] = {'centers': [], 'minority_voxels': {}, 'has_minority': False}
            except Exception as e:
                print(f"Warning: Could not load case {case_path}: {e}")
                self.case_info[case_path] = {'centers': [], 'minority_voxels': {}, 'has_minority': False}
        print(f"Pre-computed info for {len(self.case_info)} cases")
    
    def sample_patch(self, case_idx: int) -> Tuple[np.ndarray, np.ndarray, str]:
        """
        Sample patch from a case
        
        Args:
            case_idx: Index of case in self.cases
            
        Returns:
            Tuple of (image_patch, label_patch, case_id)
            image_patch: Fused 4-modality patch or single-channel fused patch
            label_patch: Label patch
            case_id: Case identifier
        """
        case_path = self.cases[case_idx]
        case_id = os.path.basename(case_path)
        
        # Load case data
        data = load_mha_case(case_path)
        
        # Determine sampling strategy
        use_tumor_center = np.random.rand() < self.tumor_ratio
        use_minority_class = np.random.rand() < self.minority_class_ratio
        
        info = self.case_info.get(case_path, {'centers': [], 'minority_voxels': {}, 'has_minority': False})
        
        # Sample center
        if use_tumor_center and len(info['centers']) > 0:
            # Tumor-centric sampling
            center = random.choice(info['centers'])
        elif use_minority_class and info['has_minority']:
            # Class-aware sampling: prioritize patches with minority classes
            minority_voxels = info['minority_voxels']
            all_minority = minority_voxels[1] + minority_voxels[3]
            if len(all_minority) > 0:
                center = random.choice(all_minority)
            elif len(info['centers']) > 0:
                center = random.choice(info['centers'])
            else:
                # Random sampling
                d, h, w = data['t1'].shape if data['t1'] is not None else (155, 240, 240)
                center = (np.random.randint(0, d), np.random.randint(0, h), np.random.randint(0, w))
        else:
            # Random sampling
            d, h, w = data['t1'].shape if data['t1'] is not None else (155, 240, 240)
            center = (np.random.randint(0, d), np.random.randint(0, h), np.random.randint(0, w))
        
        # Extract patches for each modality
        patches = {}
        for mod in ['t1', 't1ce', 't2', 'flair', 'label']:
            if data[mod] is not None:
                patches[mod] = extract_patch(data[mod], center, self.patch_size)
            else:
                patches[mod] = np.zeros(self.patch_size, dtype=np.float32)
        
        # Normalize each modality
        for mod in ['t1', 't1ce', 't2', 'flair']:
            if mod in patches:
                patches[mod] = zscore_norm(patches[mod])
        
        # Stack modalities (will be fused later in dataset)
        image_patch = np.stack([patches['t1'], patches['t1ce'], patches['t2'], patches['flair']], axis=0)
        label_patch = patches['label']
        
        return image_patch, label_patch, case_id

# Initialize sampler (if cases available)
if len(sampler_cases) > 0:
    sampler = TumorPatchSampler(
        cases=sampler_cases,
        patch_size=config.patch_size,
        tumor_ratio=config.tumor_centric_ratio,
        minority_class_ratio=config.minority_class_ratio
    )
    print(f"Sampler initialized with {len(sampler.cases)} cases (train + val)")
else:
    print("Cannot initialize sampler - no cases discovered.")


### Cell 7: Dataset Class(Shruthi)
All patches are now sampled from the entire BRATS2015 training split. The upstream 85/15 train/validation division is fixed for the run, and the validation subset remains untouched until final evaluation.


In [ ]:
# Cell 7: Dataset Class
class BRATSPatchDataset(Dataset):
    """
    Dataset class for BraTS patches with support for regular and learnable fusion
    """
    def __init__(self, cases: List[str], sampler: TumorPatchSampler, 
                 fusion_mode: str = 'mean', num_patches_per_case: int = 8):
        """
        Args:
            cases: List of case paths
            sampler: TumorPatchSampler instance
            fusion_mode: 'mean' for regular fusion, 'learnable' for 4-channel output
            num_patches_per_case: Number of patches to sample per case
        """
        self.cases = cases
        self.sampler = sampler
        self.fusion_mode = fusion_mode
        self.num_patches_per_case = num_patches_per_case
        
        # Create case index mapping
        self.case_indices = []
        for i, case_path in enumerate(cases):
            if case_path in sampler.cases:
                case_idx = sampler.cases.index(case_path)
                for _ in range(num_patches_per_case):
                    self.case_indices.append((case_idx, case_path))
        
        print(f"Dataset initialized: {len(self.case_indices)} patches from {len(cases)} cases")
        print(f"  Fusion mode: {fusion_mode}")
        print(f"  Patches per case: {num_patches_per_case}")
    
    def __len__(self):
        return len(self.case_indices)
    
    def __getitem__(self, idx):
        case_idx, case_path = self.case_indices[idx]
        
        # Sample patch
        image_patch, label_patch, case_id = self.sampler.sample_patch(case_idx)
        
        # Apply fusion based on mode
        if self.fusion_mode == 'mean':
            # Regular fusion: mean across modalities -> single channel
            image_patch = image_patch.mean(axis=0, keepdims=True)  # (1, D, H, W)
        elif self.fusion_mode == 'learnable':
            # Learnable fusion: keep 4 channels -> (4, D, H, W)
            pass  # Already in correct shape
        else:
            raise ValueError(f"Unknown fusion mode: {self.fusion_mode}")
        
        # Convert to tensors
        image_tensor = torch.from_numpy(image_patch).float()
        label_tensor = torch.from_numpy(label_patch).long()
        
        return {
            'image': image_tensor,
            'label': label_tensor,
            'case_id': case_id
        }

# Create real datasets (if cases + sampler are ready)
if len(train_cases) > 0 and len(val_cases) > 0 and 'sampler' in locals():
    configured_fusion_mode = 'learnable' if config.use_learnable_fusion else 'mean'
    train_dataset = BRATSPatchDataset(
        cases=train_cases,
        sampler=sampler,
        fusion_mode=configured_fusion_mode,
        num_patches_per_case=config.train_patches_per_case
    )
    
    val_dataset = BRATSPatchDataset(
        cases=val_cases,
        sampler=sampler,
        fusion_mode=configured_fusion_mode,
        num_patches_per_case=config.val_patches_per_case
    )
    
    print(f"\nDataset split (real data only):")
    print(f"  Training: {len(train_dataset)} patches from {len(train_cases)} cases")
    print(f"  Validation: {len(val_dataset)} patches from {len(val_cases)} cases")
else:
    print("Cannot create dataset - missing cases or sampler")


### Cell 8: DataLoader Setup(Shruthi)
These DataLoaders wrap the real datasets only; no dummy tensors are instantiated. Training uses the 85% split, and validation loaders are reserved strictly for evaluation and checkpoint selection.


In [ ]:
# Cell 8: DataLoader Setup
def create_dataloaders(train_dataset: Dataset, val_dataset: Dataset, 
                      test_dataset: Optional[Dataset], config: TrainConfig):
    """
    Create DataLoaders with appropriate settings
    
    Args:
        train_dataset: Training dataset
        val_dataset: Validation dataset
        test_dataset: Optional test dataset
        config: Training configuration
        
    Returns:
        Tuple of (train_loader, val_loader, test_loader)
    """
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=config.num_workers,
        pin_memory=False,  # Set to False for Kaggle memory constraints
        persistent_workers=False
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        pin_memory=False,
        persistent_workers=False
    )
    
    test_loader = None
    if test_dataset is not None:
        test_loader = DataLoader(
            test_dataset,
            batch_size=config.batch_size,
            shuffle=False,
            num_workers=config.num_workers,
            pin_memory=False,
            persistent_workers=False
        )
    
    return train_loader, val_loader, test_loader

def print_dataset_statistics(dataset: Dataset, name: str):
    """Print statistics about the dataset"""
    print(f"\n{name} Dataset Statistics:")
    print(f"  Total patches: {len(dataset)}")
    
    # Count class distribution
    class_counts = defaultdict(int)
    total_voxels = 0
    
    for i in range(min(100, len(dataset))):  # Sample first 100 patches
        sample = dataset[i]
        label = sample['label'].numpy()
        unique, counts = np.unique(label, return_counts=True)
        for u, c in zip(unique, counts):
            class_counts[int(u)] += c
            total_voxels += c
    
    print(f"  Class distribution (from sample):")
    class_names = {0: "Background", 1: "Necrosis", 2: "Edema", 3: "Non-enhancing", 4: "Enhancing"}
    for class_id in sorted(class_counts.keys()):
        count = class_counts[class_id]
        percentage = (count / total_voxels * 100) if total_voxels > 0 else 0
        print(f"    {class_names.get(class_id, f'Class {class_id}')}: {count} voxels ({percentage:.2f}%)")

# Create DataLoaders (if datasets exist)
if 'train_dataset' in locals() and 'val_dataset' in locals():
    train_loader, val_loader, test_loader = create_dataloaders(
        train_dataset, val_dataset, None, config
    )
    
    print(f"\nDataLoaders created:")
    print(f"  Training batches: {len(train_loader)}")
    print(f"  Validation batches: {len(val_loader)}")
    
    # Print statistics for sanity (sample first 100 patches)
    print_dataset_statistics(train_dataset, "Training")
    print_dataset_statistics(val_dataset, "Validation")
else:
    print("Cannot create DataLoaders - datasets not initialized")


## Phase 3: Model Architecture(Hrishikesh and Shruthi)

_The architectural cells below omit standalone dummy forward passes; the blocks are defined once and consumed by the real training/evaluation loops._

### Cell 9: Residual Block(Hrishikesh)


In [ ]:
# Cell 9: Residual Block
class ResidualBlock3d(nn.Module):
    """
    3D Residual Block with InstanceNorm3d for small batch training
    
    Architecture:
        Input -> Conv3d -> InstanceNorm3d -> ReLU -> Conv3d -> InstanceNorm3d -> (+residual) -> ReLU -> Output
    
    Uses InstanceNorm3d instead of BatchNorm3d for stable training with batch_size=1-2
    """
    def __init__(self, in_channels: int, out_channels: int, 
                 use_instance_norm: bool = True, dropout_rate: float = 0.0):
        """
        Args:
            in_channels: Number of input channels
            out_channels: Number of output channels
            use_instance_norm: Use InstanceNorm3d (True) or BatchNorm3d (False)
            dropout_rate: Dropout rate (0.0 = no dropout)
        """
        super(ResidualBlock3d, self).__init__()
        
        self.use_instance_norm = use_instance_norm
        self.dropout_rate = dropout_rate
        
        # First convolution
        self.conv1 = nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
        
        # Normalization layer
        if use_instance_norm:
            self.norm1 = nn.InstanceNorm3d(out_channels, affine=True)
            self.norm2 = nn.InstanceNorm3d(out_channels, affine=True)
        else:
            self.norm1 = nn.BatchNorm3d(out_channels)
            self.norm2 = nn.BatchNorm3d(out_channels)
        
        # Second convolution
        self.conv2 = nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        
        # ReLU activation
        self.relu = nn.ReLU(inplace=True)
        
        # Dropout (optional)
        self.dropout = nn.Dropout3d(p=dropout_rate) if dropout_rate > 0 else None
        
        # Residual connection (1x1 conv if channel mismatch)
        if in_channels != out_channels:
            self.residual_conv = nn.Sequential(
                nn.Conv3d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.InstanceNorm3d(out_channels, affine=True) if use_instance_norm else nn.BatchNorm3d(out_channels)
            )
        else:
            self.residual_conv = None
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass
        
        Args:
            x: Input tensor (B, C, D, H, W)
            
        Returns:
            Output tensor (B, out_channels, D, H, W)
        """
        # Store residual
        residual = x
        
        # First conv block
        out = self.conv1(x)
        out = self.norm1(out)
        out = self.relu(out)
        
        # Second conv block
        out = self.conv2(out)
        out = self.norm2(out)
        
        # Apply residual connection
        if self.residual_conv is not None:
            residual = self.residual_conv(residual)
        
        out = out + residual
        out = self.relu(out)
        
        # Apply dropout if specified
        if self.dropout is not None:
            out = self.dropout(out)
        
        return out



### Cell 10: Encoder and Decoder Blocks(Hrishikesh)


In [ ]:
# Cell 10: Encoder and Decoder Blocks
class DownBlock(nn.Module):
    """
    Encoder downsampling block
    
    Architecture:
        Input -> MaxPool3d(2) -> ResidualBlock3d -> Output
    
    Dropout is applied based on encoder level (deeper = more dropout)
    """
    def __init__(self, in_channels: int, out_channels: int, 
                 use_instance_norm: bool = True, dropout_rate: float = 0.0):
        """
        Args:
            in_channels: Number of input channels
            out_channels: Number of output channels
            use_instance_norm: Use InstanceNorm3d (True) or BatchNorm3d (False)
            dropout_rate: Dropout rate for this level
        """
        super(DownBlock, self).__init__()
        
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2)
        self.residual_block = ResidualBlock3d(
            in_channels=in_channels,
            out_channels=out_channels,
            use_instance_norm=use_instance_norm,
            dropout_rate=dropout_rate
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass
        
        Args:
            x: Input tensor (B, C, D, H, W)
            
        Returns:
            Downsampled tensor (B, out_channels, D/2, H/2, W/2)
        """
        x = self.pool(x)
        x = self.residual_block(x)
        return x


class UpBlock(nn.Module):
    """
    Decoder upsampling block with skip connection
    
    Architecture:
        Input -> ConvTranspose3d(stride=2) -> Concat(skip) -> ResidualBlock3d -> Output
    
    Skip connections from encoder help preserve spatial details for boundary segmentation
    """
    def __init__(self, in_channels: int, out_channels: int, 
                 use_instance_norm: bool = True, dropout_rate: float = 0.0):
        """
        Args:
            in_channels: Number of input channels (from lower level)
            out_channels: Number of output channels
            use_instance_norm: Use InstanceNorm3d (True) or BatchNorm3d (False)
            dropout_rate: Dropout rate for this level
        """
        super(UpBlock, self).__init__()
        
        # Transposed convolution for upsampling
        self.up_conv = nn.ConvTranspose3d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=2,
            stride=2
        )
        
        # Residual block after concatenation (in_channels = out_channels + skip_channels)
        # After concat: out_channels (from up) + out_channels (from skip) = 2 * out_channels
        self.residual_block = ResidualBlock3d(
            in_channels=out_channels * 2,  # After concatenation with skip
            out_channels=out_channels,
            use_instance_norm=use_instance_norm,
            dropout_rate=dropout_rate
        )
    
    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        """
        Forward pass with skip connection
        
        Args:
            x: Input tensor from lower level (B, C, D, H, W)
            skip: Skip connection tensor from encoder (B, out_channels, D*2, H*2, W*2)
            
        Returns:
            Upsampled tensor (B, out_channels, D*2, H*2, W*2)
        """
        # Upsample
        x = self.up_conv(x)
        
        # Handle size mismatch (due to odd dimensions)
        if x.shape != skip.shape:
            # Pad x to match skip dimensions
            diff_d = skip.shape[2] - x.shape[2]
            diff_h = skip.shape[3] - x.shape[3]
            diff_w = skip.shape[4] - x.shape[4]
            x = F.pad(x, [
                diff_w // 2, diff_w - diff_w // 2,
                diff_h // 2, diff_h - diff_h // 2,
                diff_d // 2, diff_d - diff_d // 2
            ])
        
        # Concatenate with skip connection
        x = torch.cat([x, skip], dim=1)
        
        # Apply residual block
        x = self.residual_block(x)
        
        return x



### Cell 11: Res-UNet 3D Architecture(Hrishikesh)


In [ ]:
# Cell 11: Res-UNet 3D Architecture
class ResUNet3D(nn.Module):
    """
    3D Residual U-Net for brain tumor segmentation
    
    Architecture:
        - Encoder: 4 levels with filters [16, 32, 64, 128]
        - Bottleneck: 256 filters with Dropout(0.25)
        - Decoder: 4 levels with skip connections
        - Output: Conv3d to num_classes (no softmax - applied in loss)
    
    Dropout is applied progressively:
        - Levels 1-2: No dropout (shallow features)
        - Levels 3-4: Dropout(0.25) (deeper features)
        - Bottleneck: Dropout(0.25)
    
    Uses InstanceNorm3d for stable training with batch_size=1-2
    """
    def __init__(self, in_channels: int = 1, num_classes: int = 5, 
                 base_filters: int = 16, depth: int = 4,
                 use_instance_norm: bool = True,
                 dropout_rate_deep: float = 0.25,
                 dropout_rate_shallow: float = 0.0):
        """
        Args:
            in_channels: Number of input channels (1 for fused, 4 for learnable fusion)
            num_classes: Number of output classes (5 for BraTS: bg + 4 tumor classes)
            base_filters: Number of filters in first level (doubles each level)
            depth: Number of encoder/decoder levels
            use_instance_norm: Use InstanceNorm3d (True) or BatchNorm3d (False)
            dropout_rate_deep: Dropout rate for deeper levels (3-4) and bottleneck
            dropout_rate_shallow: Dropout rate for shallow levels (1-2)
        """
        super(ResUNet3D, self).__init__()
        
        self.in_channels = in_channels
        self.num_classes = num_classes
        self.base_filters = base_filters
        self.depth = depth
        self.use_instance_norm = use_instance_norm
        
        # Calculate filter sizes for each level
        # Level 0: base_filters, Level 1: base_filters*2, etc.
        self.filters = [base_filters * (2 ** i) for i in range(depth + 1)]
        # filters = [16, 32, 64, 128, 256] for base_filters=16, depth=4
        
        # Dropout rates for each level
        self.dropout_rates = []
        for i in range(depth):
            if i < 2:  # Shallow levels (1-2)
                self.dropout_rates.append(dropout_rate_shallow)
            else:  # Deeper levels (3-4)
                self.dropout_rates.append(dropout_rate_deep)
        
        # ============ ENCODER ============
        # Initial convolution (input to first level)
        self.initial_conv = ResidualBlock3d(
            in_channels=in_channels,
            out_channels=self.filters[0],
            use_instance_norm=use_instance_norm,
            dropout_rate=0.0  # No dropout at input
        )
        
        # Encoder blocks
        self.encoder_blocks = nn.ModuleList()
        for i in range(depth):
            self.encoder_blocks.append(
                DownBlock(
                    in_channels=self.filters[i],
                    out_channels=self.filters[i + 1],
                    use_instance_norm=use_instance_norm,
                    dropout_rate=self.dropout_rates[i]
                )
            )
        
        # ============ BOTTLENECK ============
        # Additional processing at lowest level
        self.bottleneck = nn.Sequential(
            ResidualBlock3d(
                in_channels=self.filters[-1],
                out_channels=self.filters[-1],
                use_instance_norm=use_instance_norm,
                dropout_rate=dropout_rate_deep
            )
        )
        
        # ============ DECODER ============
        # Decoder blocks (reverse order of encoder)
        self.decoder_blocks = nn.ModuleList()
        for i in range(depth - 1, -1, -1):  # depth-1, depth-2, ..., 0
            # Determine dropout for decoder level (mirror encoder)
            if i < 2:
                decoder_dropout = dropout_rate_shallow
            else:
                decoder_dropout = dropout_rate_deep
            
            self.decoder_blocks.append(
                UpBlock(
                    in_channels=self.filters[i + 1],
                    out_channels=self.filters[i],
                    use_instance_norm=use_instance_norm,
                    dropout_rate=decoder_dropout
                )
            )
        
        # ============ OUTPUT ============
        # Final convolution to num_classes
        self.output_conv = nn.Conv3d(
            in_channels=self.filters[0],
            out_channels=num_classes,
            kernel_size=1
        )
        
        # Initialize weights
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Initialize model weights using He initialization"""
        for m in self.modules():
            if isinstance(m, nn.Conv3d) or isinstance(m, nn.ConvTranspose3d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, (nn.BatchNorm3d, nn.InstanceNorm3d)):
                if m.weight is not None:
                    nn.init.constant_(m.weight, 1)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass
        
        Args:
            x: Input tensor (B, in_channels, D, H, W)
            
        Returns:
            Output logits (B, num_classes, D, H, W) - no softmax applied
        """
        # Store skip connections
        skips = []
        
        # ============ ENCODER ============
        # Initial convolution
        x = self.initial_conv(x)
        skips.append(x)  # Skip for first decoder level
        
        # Encoder blocks
        for i, encoder_block in enumerate(self.encoder_blocks):
            x = encoder_block(x)
            if i < len(self.encoder_blocks) - 1:  # Don't store last as skip
                skips.append(x)
        
        # ============ BOTTLENECK ============
        x = self.bottleneck(x)
        
        # ============ DECODER ============
        # Reverse skips for decoder
        skips = skips[::-1]
        
        for i, decoder_block in enumerate(self.decoder_blocks):
            x = decoder_block(x, skips[i])
        
        # ============ OUTPUT ============
        x = self.output_conv(x)
        
        return x
    
    def get_num_parameters(self) -> int:
        """Get total number of trainable parameters"""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)
    
    def print_summary(self):
        """Print model summary"""
        print(f"ResUNet3D Summary:")
        print(f"  Input channels: {self.in_channels}")
        print(f"  Output classes: {self.num_classes}")
        print(f"  Base filters: {self.base_filters}")
        print(f"  Depth: {self.depth}")
        print(f"  Filter sizes: {self.filters}")
        print(f"  Dropout rates (encoder): {self.dropout_rates}")
        print(f"  Use InstanceNorm: {self.use_instance_norm}")
        print(f"  Total parameters: {self.get_num_parameters():,}")



### Cell 12: Learnable Fusion Module(Shruthi)


In [ ]:
# Cell 12: Learnable Fusion Module
class LearnableFusionResUNet3D(nn.Module):
    """
    ResUNet3D with learnable modality fusion weights
    
    This module wraps ResUNet3D and adds learnable weights for fusing
    the 4 MRI modalities (T1, T1ce, T2, FLAIR) before passing to the network.
    
    The fusion weights are learned during training and can adapt to
    emphasize modalities that are more informative for segmentation.
    
    Experiments showed 79% improvement in Dice (0.3262 vs 0.1824) with
    learnable fusion compared to mean fusion.
    """
    def __init__(self, num_classes: int = 5, base_filters: int = 16, 
                 depth: int = 4, use_instance_norm: bool = True,
                 dropout_rate_deep: float = 0.25, dropout_rate_shallow: float = 0.0):
        """
        Args:
            num_classes: Number of output classes (5 for BraTS)
            base_filters: Number of filters in first level
            depth: Number of encoder/decoder levels
            use_instance_norm: Use InstanceNorm3d (True) or BatchNorm3d (False)
            dropout_rate_deep: Dropout rate for deeper levels
            dropout_rate_shallow: Dropout rate for shallow levels
        """
        super(LearnableFusionResUNet3D, self).__init__()
        
        self.num_modalities = 4  # T1, T1ce, T2, FLAIR
        self.num_classes = num_classes
        
        # Learnable fusion weights (initialized to equal weights)
        # Using log-scale to ensure positive weights after softmax
        self.fusion_weights = nn.Parameter(torch.zeros(self.num_modalities))
        
        # ResUNet3D backbone (takes single fused channel as input)
        self.backbone = ResUNet3D(
            in_channels=1,  # Single fused channel
            num_classes=num_classes,
            base_filters=base_filters,
            depth=depth,
            use_instance_norm=use_instance_norm,
            dropout_rate_deep=dropout_rate_deep,
            dropout_rate_shallow=dropout_rate_shallow
        )
    
    def get_fusion_weights(self) -> torch.Tensor:
        """
        Get normalized fusion weights (sum to 1)
        
        Returns:
            Normalized weights for each modality
        """
        return F.softmax(self.fusion_weights, dim=0)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass with learnable fusion
        
        Args:
            x: Input tensor (B, 4, D, H, W) with 4 modalities
            
        Returns:
            Output logits (B, num_classes, D, H, W)
        """
        # Get normalized weights
        weights = self.get_fusion_weights()  # (4,)
        
        # Apply weighted fusion
        # x: (B, 4, D, H, W) -> (B, 1, D, H, W)
        # Reshape weights for broadcasting: (4,) -> (1, 4, 1, 1, 1)
        weights = weights.view(1, self.num_modalities, 1, 1, 1)
        
        # Weighted sum across modalities
        fused = (x * weights).sum(dim=1, keepdim=True)  # (B, 1, D, H, W)
        
        # Pass through backbone
        output = self.backbone(fused)
        
        return output
    
    def print_fusion_weights(self):
        """Print current fusion weights"""
        weights = self.get_fusion_weights().detach().cpu().numpy()
        modality_names = ['T1', 'T1ce', 'T2', 'FLAIR']
        print("Current fusion weights:")
        for name, weight in zip(modality_names, weights):
            print(f"  {name}: {weight:.4f}")
    
    def get_num_parameters(self) -> int:
        """Get total number of trainable parameters"""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)
    
    def print_summary(self):
        """Print model summary"""
        print(f"LearnableFusionResUNet3D Summary:")
        print(f"  Number of modalities: {self.num_modalities}")
        print(f"  Output classes: {self.num_classes}")
        self.print_fusion_weights()
        print(f"  Backbone parameters: {self.backbone.get_num_parameters():,}")
        print(f"  Fusion parameters: {self.num_modalities}")
        print(f"  Total parameters: {self.get_num_parameters():,}")



## Phase 4: Loss Functions and Metrics(Hrishikesh and Nihal)

### Cell 13: Loss Functions(Hrishikesh)


In [ ]:
# Cell 13: Loss Functions
def dice_loss(pred: torch.Tensor, target: torch.Tensor, 
              num_classes: int = 5, smooth: float = 1e-4) -> torch.Tensor:
    """
    Multi-class Soft Dice Loss
    
    Dice loss is effective for handling class imbalance and focuses on
    overlap between prediction and ground truth.
    
    Args:
        pred: Predicted logits (B, num_classes, D, H, W)
        target: Ground truth labels (B, D, H, W) with class indices
        num_classes: Number of classes
        smooth: Smoothing factor to prevent division by zero
        
    Returns:
        Dice loss (scalar)
    """
    # Apply softmax to get probabilities
    pred_soft = F.softmax(pred, dim=1)  # (B, C, D, H, W)
    
    # One-hot encode target
    target_one_hot = F.one_hot(target.long(), num_classes)  # (B, D, H, W, C)
    target_one_hot = target_one_hot.permute(0, 4, 1, 2, 3).float()  # (B, C, D, H, W)
    
    # Calculate Dice for each class
    dice_scores = []
    for c in range(num_classes):
        pred_c = pred_soft[:, c]  # (B, D, H, W)
        target_c = target_one_hot[:, c]  # (B, D, H, W)
        
        intersection = (pred_c * target_c).sum()
        union = pred_c.sum() + target_c.sum()
        
        dice = (2.0 * intersection + smooth) / (union + smooth)
        dice_scores.append(dice)
    
    # Average Dice across classes
    mean_dice = torch.stack(dice_scores).mean()
    
    # Return 1 - Dice as loss
    return 1.0 - mean_dice


def focal_loss(pred: torch.Tensor, target: torch.Tensor,
               alpha: List[float] = None, gamma: float = 2.0) -> torch.Tensor:
    """
    Focal Loss for multi-class classification
    
    Focal loss down-weights easy examples and focuses on hard examples,
    which is crucial for handling class imbalance in tumor segmentation.
    
    Alpha weights: [0.1, 2.0, 2.0, 3.0, 3.0] for [bg, necrosis, edema, non-enh, enh]
    
    Args:
        pred: Predicted logits (B, num_classes, D, H, W)
        target: Ground truth labels (B, D, H, W) with class indices
        alpha: Per-class weights (higher = more importance)
        gamma: Focusing parameter (higher = more focus on hard examples)
        
    Returns:
        Focal loss (scalar)
    """
    num_classes = pred.shape[1]
    
    # Default alpha weights if not provided
    if alpha is None:
        alpha = [0.1, 2.0, 2.0, 3.0, 3.0]  # [bg, necrosis, edema, non-enh, enh]
    
    # Convert alpha to tensor
    alpha_tensor = torch.tensor(alpha, dtype=pred.dtype, device=pred.device)
    
    # Apply softmax to get probabilities
    pred_soft = F.softmax(pred, dim=1)  # (B, C, D, H, W)
    
    # Reshape for cross entropy: (B, C, D, H, W) -> (B*D*H*W, C)
    pred_flat = pred.permute(0, 2, 3, 4, 1).contiguous().view(-1, num_classes)
    target_flat = target.view(-1).long()
    
    # Get predicted probability for correct class
    pt = pred_soft.permute(0, 2, 3, 4, 1).contiguous().view(-1, num_classes)
    pt = pt.gather(1, target_flat.unsqueeze(1)).squeeze(1)  # (B*D*H*W,)
    
    # Get alpha weight for each sample
    alpha_t = alpha_tensor[target_flat]  # (B*D*H*W,)
    
    # Compute focal weight: (1 - pt)^gamma
    focal_weight = (1.0 - pt) ** gamma
    
    # Cross entropy loss
    ce_loss = F.cross_entropy(pred_flat, target_flat, reduction='none')
    
    # Apply focal weight and alpha
    focal_loss_value = alpha_t * focal_weight * ce_loss
    
    return focal_loss_value.mean()


def combined_dice_focal_loss(pred: torch.Tensor, target: torch.Tensor,
                             alpha: List[float] = None, gamma: float = 2.0,
                             dice_weight: float = 0.4, focal_weight: float = 0.6,
                             num_classes: int = 5) -> torch.Tensor:
    """
    Combined Dice + Focal Loss
    
    Experiments showed that Dice+Focal prevents class collapse that occurs
    with Dice-only loss. Recommended weights: dice_weight=0.4, focal_weight=0.6
    
    Args:
        pred: Predicted logits (B, num_classes, D, H, W)
        target: Ground truth labels (B, D, H, W)
        alpha: Per-class weights for focal loss
        gamma: Focusing parameter for focal loss
        dice_weight: Weight for Dice loss component
        focal_weight: Weight for Focal loss component
        num_classes: Number of classes
        
    Returns:
        Combined loss (scalar)
    """
    # Compute individual losses
    d_loss = dice_loss(pred, target, num_classes=num_classes)
    f_loss = focal_loss(pred, target, alpha=alpha, gamma=gamma)
    
    # Combine losses
    combined = dice_weight * d_loss + focal_weight * f_loss
    
    return combined


class EMALoss:
    """
    Exponential Moving Average for loss smoothing
    
    Helps reduce noise in loss values for more stable training monitoring
    """
    def __init__(self, alpha: float = 0.1):
        """
        Args:
            alpha: Smoothing factor (0 = no smoothing, 1 = no memory)
        """
        self.alpha = alpha
        self.ema_value = None
    
    def update(self, loss_value: float) -> float:
        """
        Update EMA with new loss value
        
        Args:
            loss_value: Current loss value
            
        Returns:
            Smoothed loss value
        """
        if self.ema_value is None:
            self.ema_value = loss_value
        else:
            self.ema_value = self.alpha * loss_value + (1 - self.alpha) * self.ema_value
        return self.ema_value
    
    def reset(self):
        """Reset EMA"""
        self.ema_value = None


print("Loss functions ready for use in the real training loop. Instantiate dataloaders in Phase 3, then call these losses from train_epoch/validate_epoch without any synthetic inputs.")


### Cell 14: Evaluation Metrics(Hrishikesh)


In [ ]:
# Cell 14: Evaluation Metrics
def dice_score(pred: torch.Tensor, target: torch.Tensor, 
               num_classes: int = 5, smooth: float = 1e-4) -> Dict[int, float]:
    """
    Compute per-class Dice scores
    
    Args:
        pred: Predicted logits (B, num_classes, D, H, W) or class indices (B, D, H, W)
        target: Ground truth labels (B, D, H, W)
        num_classes: Number of classes
        smooth: Smoothing factor
        
    Returns:
        Dictionary of {class_id: dice_score}
    """
    # Handle logits vs class indices
    if pred.dim() == 5:
        pred_classes = pred.argmax(dim=1)  # (B, D, H, W)
    else:
        pred_classes = pred
    
    dice_scores = {}
    
    for c in range(num_classes):
        pred_c = (pred_classes == c).float()
        target_c = (target == c).float()
        
        intersection = (pred_c * target_c).sum().item()
        union = pred_c.sum().item() + target_c.sum().item()
        
        if union > 0:
            dice = (2.0 * intersection + smooth) / (union + smooth)
        else:
            dice = 1.0 if intersection == 0 else 0.0  # Both empty = perfect match
        
        dice_scores[c] = dice
    
    return dice_scores


def macro_tumor_dice(dice_scores: Dict[int, float]) -> float:
    """
    Compute macro-averaged Dice for tumor classes (1-4)
    
    This is the PRIMARY METRIC for evaluation - average of all tumor class Dice scores
    
    Args:
        dice_scores: Per-class Dice scores from dice_score()
        
    Returns:
        Macro-averaged Dice for tumor classes
    """
    tumor_dices = [dice_scores.get(c, 0.0) for c in range(1, 5)]  # Classes 1, 2, 3, 4
    return sum(tumor_dices) / len(tumor_dices)


def compute_wt_tc_et_dice(pred: torch.Tensor, target: torch.Tensor, 
                          smooth: float = 1e-4) -> Dict[str, float]:
    """
    Compute Whole Tumor, Tumor Core, and Enhancing Tumor Dice scores
    
    These are the standard BraTS challenge metrics:
    - WT (Whole Tumor): All tumor classes (1, 2, 3, 4)
    - TC (Tumor Core): Necrosis + Non-enhancing + Enhancing (1, 3, 4)
    - ET (Enhancing Tumor): Enhancing only (4)
    
    Args:
        pred: Predicted logits (B, num_classes, D, H, W) or class indices (B, D, H, W)
        target: Ground truth labels (B, D, H, W)
        smooth: Smoothing factor
        
    Returns:
        Dictionary with 'WT', 'TC', 'ET' Dice scores
    """
    # Handle logits vs class indices
    if pred.dim() == 5:
        pred_classes = pred.argmax(dim=1)
    else:
        pred_classes = pred
    
    results = {}
    
    # Whole Tumor (WT): classes 1, 2, 3, 4
    pred_wt = ((pred_classes == 1) | (pred_classes == 2) | 
               (pred_classes == 3) | (pred_classes == 4)).float()
    target_wt = ((target == 1) | (target == 2) | 
                 (target == 3) | (target == 4)).float()
    
    intersection_wt = (pred_wt * target_wt).sum().item()
    union_wt = pred_wt.sum().item() + target_wt.sum().item()
    results['WT'] = (2.0 * intersection_wt + smooth) / (union_wt + smooth) if union_wt > 0 else 1.0
    
    # Tumor Core (TC): classes 1, 3, 4 (necrosis, non-enhancing, enhancing)
    pred_tc = ((pred_classes == 1) | (pred_classes == 3) | (pred_classes == 4)).float()
    target_tc = ((target == 1) | (target == 3) | (target == 4)).float()
    
    intersection_tc = (pred_tc * target_tc).sum().item()
    union_tc = pred_tc.sum().item() + target_tc.sum().item()
    results['TC'] = (2.0 * intersection_tc + smooth) / (union_tc + smooth) if union_tc > 0 else 1.0
    
    # Enhancing Tumor (ET): class 4 only
    pred_et = (pred_classes == 4).float()
    target_et = (target == 4).float()
    
    intersection_et = (pred_et * target_et).sum().item()
    union_et = pred_et.sum().item() + target_et.sum().item()
    results['ET'] = (2.0 * intersection_et + smooth) / (union_et + smooth) if union_et > 0 else 1.0
    
    return results


def iou_score(pred: torch.Tensor, target: torch.Tensor, 
              num_classes: int = 5, smooth: float = 1e-4) -> Dict[int, float]:
    """
    Compute per-class Intersection over Union (IoU) scores
    
    Args:
        pred: Predicted logits (B, num_classes, D, H, W) or class indices (B, D, H, W)
        target: Ground truth labels (B, D, H, W)
        num_classes: Number of classes
        smooth: Smoothing factor
        
    Returns:
        Dictionary of {class_id: iou_score}
    """
    # Handle logits vs class indices
    if pred.dim() == 5:
        pred_classes = pred.argmax(dim=1)
    else:
        pred_classes = pred
    
    iou_scores = {}
    
    for c in range(num_classes):
        pred_c = (pred_classes == c).float()
        target_c = (target == c).float()
        
        intersection = (pred_c * target_c).sum().item()
        union = pred_c.sum().item() + target_c.sum().item() - intersection
        
        if union > 0:
            iou = (intersection + smooth) / (union + smooth)
        else:
            iou = 1.0 if intersection == 0 else 0.0
        
        iou_scores[c] = iou
    
    return iou_scores


def sensitivity_specificity(pred: torch.Tensor, target: torch.Tensor,
                           num_classes: int = 5) -> Dict[int, Dict[str, float]]:
    """
    Compute per-class sensitivity (recall) and specificity
    
    Sensitivity = TP / (TP + FN) - How well we detect the class
    Specificity = TN / (TN + FP) - How well we avoid false positives
    
    Args:
        pred: Predicted logits (B, num_classes, D, H, W) or class indices (B, D, H, W)
        target: Ground truth labels (B, D, H, W)
        num_classes: Number of classes
        
    Returns:
        Dictionary of {class_id: {'sensitivity': value, 'specificity': value}}
    """
    # Handle logits vs class indices
    if pred.dim() == 5:
        pred_classes = pred.argmax(dim=1)
    else:
        pred_classes = pred
    
    results = {}
    
    for c in range(num_classes):
        pred_c = (pred_classes == c)
        target_c = (target == c)
        
        # True Positives, False Positives, True Negatives, False Negatives
        tp = (pred_c & target_c).sum().item()
        fp = (pred_c & ~target_c).sum().item()
        tn = (~pred_c & ~target_c).sum().item()
        fn = (~pred_c & target_c).sum().item()
        
        # Sensitivity (True Positive Rate / Recall)
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 1.0
        
        # Specificity (True Negative Rate)
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 1.0
        
        results[c] = {
            'sensitivity': sensitivity,
            'specificity': specificity
        }
    
    return results


def print_metrics(dice_scores: Dict[int, float], 
                  wt_tc_et: Dict[str, float],
                  iou_scores: Dict[int, float] = None,
                  sens_spec: Dict[int, Dict[str, float]] = None):
    """Pretty print all evaluation metrics"""
    class_names = {0: "Background", 1: "Necrosis", 2: "Edema", 3: "Non-enhancing", 4: "Enhancing"}
    
    print("\n" + "="*60)
    print("EVALUATION METRICS")
    print("="*60)
    
    # Per-class Dice
    print("\nPer-class Dice Scores:")
    for c, score in dice_scores.items():
        print(f"  {class_names.get(c, f'Class {c}')}: {score:.4f}")
    
    # Macro tumor Dice
    macro_dice = macro_tumor_dice(dice_scores)
    print(f"\n  Macro Tumor Dice (PRIMARY): {macro_dice:.4f}")
    
    # WT/TC/ET Dice
    print("\nRegion-based Dice Scores:")
    print(f"  Whole Tumor (WT): {wt_tc_et['WT']:.4f}")
    print(f"  Tumor Core (TC): {wt_tc_et['TC']:.4f}")
    print(f"  Enhancing Tumor (ET): {wt_tc_et['ET']:.4f}")
    
    # IoU
    if iou_scores:
        print("\nPer-class IoU Scores:")
        for c, score in iou_scores.items():
            print(f"  {class_names.get(c, f'Class {c}')}: {score:.4f}")
    
    # Sensitivity/Specificity
    if sens_spec:
        print("\nPer-class Sensitivity/Specificity:")
        for c, metrics in sens_spec.items():
            print(f"  {class_names.get(c, f'Class {c}')}: "
                  f"Sens={metrics['sensitivity']:.4f}, Spec={metrics['specificity']:.4f}")
    
    print("="*60)


print("Metrics helpers initialized. They are invoked inside eval_model/validate_epoch using real predictions and labels sourced from the Phase 3 dataloaders.")


### Cell 15: Robust Evaluation Function(Nihal)


In [ ]:
# Cell 15: Robust Evaluation Function
@torch.no_grad()
def eval_model(model: nn.Module, loader: DataLoader, device: torch.device,
               num_classes: int = 5, num_batches: int = 15,
               use_center_crop: bool = True, crop_size: int = 48) -> Dict[str, any]:
    """
    Robust multi-batch evaluation function
    
    Uses deterministic multi-batch evaluation for consistent metrics.
    This addresses the problem of noisy single-batch evaluation.
    
    Args:
        model: PyTorch model to evaluate
        loader: DataLoader for evaluation data
        device: Device to run evaluation on
        num_classes: Number of classes
        num_batches: Number of batches to evaluate (default 15)
        use_center_crop: Whether to center-crop patches for consistency
        crop_size: Size of center crop (if use_center_crop=True)
        
    Returns:
        Comprehensive metrics dictionary with:
        - per_class_dice: Dict[int, float]
        - macro_tumor_dice: float
        - wt_tc_et: Dict[str, float]
        - per_class_iou: Dict[int, float]
        - per_class_sens_spec: Dict[int, Dict[str, float]]
        - loss: float (if loss_fn provided)
    """
    model.eval()
    
    # Accumulators for metrics
    all_dice_scores = defaultdict(list)
    all_iou_scores = defaultdict(list)
    all_sens_spec = defaultdict(lambda: {'sensitivity': [], 'specificity': []})
    all_wt_tc_et = {'WT': [], 'TC': [], 'ET': []}
    
    batch_count = 0
    
    for batch in loader:
        if batch_count >= num_batches:
            break
        
        images = batch['image'].to(device)
        labels = batch['label'].to(device)
        
        # Center crop if specified (for consistent evaluation)
        if use_center_crop and images.shape[2] > crop_size:
            # Calculate crop indices
            d, h, w = images.shape[2], images.shape[3], images.shape[4]
            d_start = (d - crop_size) // 2
            h_start = (h - crop_size) // 2
            w_start = (w - crop_size) // 2
            
            images = images[:, :, d_start:d_start+crop_size, 
                           h_start:h_start+crop_size, 
                           w_start:w_start+crop_size]
            labels = labels[:, d_start:d_start+crop_size,
                          h_start:h_start+crop_size,
                          w_start:w_start+crop_size]
        
        # Forward pass
        outputs = model(images)
        
        # Compute metrics for this batch
        dice = dice_score(outputs, labels, num_classes=num_classes)
        iou = iou_score(outputs, labels, num_classes=num_classes)
        sens_spec = sensitivity_specificity(outputs, labels, num_classes=num_classes)
        wt_tc_et = compute_wt_tc_et_dice(outputs, labels)
        
        # Accumulate
        for c in range(num_classes):
            all_dice_scores[c].append(dice[c])
            all_iou_scores[c].append(iou[c])
            all_sens_spec[c]['sensitivity'].append(sens_spec[c]['sensitivity'])
            all_sens_spec[c]['specificity'].append(sens_spec[c]['specificity'])
        
        for key in ['WT', 'TC', 'ET']:
            all_wt_tc_et[key].append(wt_tc_et[key])
        
        batch_count += 1
    
    # Average metrics across batches
    avg_dice = {c: np.mean(scores) for c, scores in all_dice_scores.items()}
    avg_iou = {c: np.mean(scores) for c, scores in all_iou_scores.items()}
    avg_sens_spec = {
        c: {
            'sensitivity': np.mean(metrics['sensitivity']),
            'specificity': np.mean(metrics['specificity'])
        }
        for c, metrics in all_sens_spec.items()
    }
    avg_wt_tc_et = {key: np.mean(values) for key, values in all_wt_tc_et.items()}
    
    # Compute macro tumor dice
    avg_macro_tumor_dice = macro_tumor_dice(avg_dice)
    
    # Compile results
    results = {
        'per_class_dice': avg_dice,
        'macro_tumor_dice': avg_macro_tumor_dice,
        'wt_tc_et': avg_wt_tc_et,
        'per_class_iou': avg_iou,
        'per_class_sens_spec': avg_sens_spec,
        'num_batches_evaluated': batch_count
    }
    
    return results


def print_eval_results(results: Dict[str, any], prefix: str = ""):
    """Pretty print evaluation results"""
    class_names = {0: "Background", 1: "Necrosis", 2: "Edema", 3: "Non-enhancing", 4: "Enhancing"}
    
    print(f"\n{prefix}{'='*60}")
    print(f"{prefix}EVALUATION RESULTS ({results['num_batches_evaluated']} batches)")
    print(f"{prefix}{'='*60}")
    
    # Primary metric
    print(f"\n{prefix}PRIMARY METRIC:")
    print(f"{prefix}  Macro Tumor Dice: {results['macro_tumor_dice']:.4f}")
    
    # Per-class Dice
    print(f"\n{prefix}Per-class Dice Scores:")
    for c, score in results['per_class_dice'].items():
        print(f"{prefix}  {class_names.get(c, f'Class {c}')}: {score:.4f}")
    
    # WT/TC/ET
    print(f"\n{prefix}Region-based Dice (WT/TC/ET):")
    print(f"{prefix}  Whole Tumor (WT): {results['wt_tc_et']['WT']:.4f}")
    print(f"{prefix}  Tumor Core (TC): {results['wt_tc_et']['TC']:.4f}")
    print(f"{prefix}  Enhancing Tumor (ET): {results['wt_tc_et']['ET']:.4f}")
    
    # Per-class IoU
    print(f"\n{prefix}Per-class IoU Scores:")
    for c, score in results['per_class_iou'].items():
        print(f"{prefix}  {class_names.get(c, f'Class {c}')}: {score:.4f}")
    
    print(f"{prefix}{'='*60}")


print("Call eval_model with the actual validation loader from Phase 3 to obtain deterministic metrics. No dummy loaders are instantiated in this notebook anymore.")


## Phase 5: Training Pipeline(Hrishikesh)

### Cell 16: Training Setup(Hrishikesh)


In [ ]:
# Cell 16: Training Setup
from torch.cuda.amp import GradScaler, autocast

class GradientNormTracker:
    """
    Track gradient norms for monitoring training stability
    
    High gradient norms indicate instability, while consistently low norms
    may indicate vanishing gradients.
    """
    def __init__(self, window_size: int = 100):
        self.window_size = window_size
        self.grad_norms = []
    
    def update(self, grad_norm: float):
        """Add new gradient norm"""
        self.grad_norms.append(grad_norm)
        if len(self.grad_norms) > self.window_size:
            self.grad_norms.pop(0)
    
    def get_stats(self) -> Dict[str, float]:
        """Get gradient norm statistics"""
        if not self.grad_norms:
            return {'mean': 0.0, 'max': 0.0, 'min': 0.0, 'std': 0.0}
        return {
            'mean': np.mean(self.grad_norms),
            'max': np.max(self.grad_norms),
            'min': np.min(self.grad_norms),
            'std': np.std(self.grad_norms)
        }
    
    def reset(self):
        """Reset tracker"""
        self.grad_norms = []


def create_model(config: TrainConfig, device: torch.device) -> nn.Module:
    """
    Create and initialize model based on configuration
    
    Args:
        config: Training configuration
        device: Device to place model on
        
    Returns:
        Initialized model
    """
    if config.use_learnable_fusion:
        model = LearnableFusionResUNet3D(
            num_classes=config.num_classes,
            base_filters=config.base_filters,
            depth=config.depth,
            use_instance_norm=config.use_instance_norm,
            dropout_rate_deep=config.dropout_rate_deep,
            dropout_rate_shallow=config.dropout_rate_shallow
        )
        print("Created LearnableFusionResUNet3D model")
    else:
        model = ResUNet3D(
            in_channels=1,  # Single fused channel for regular fusion
            num_classes=config.num_classes,
            base_filters=config.base_filters,
            depth=config.depth,
            use_instance_norm=config.use_instance_norm,
            dropout_rate_deep=config.dropout_rate_deep,
            dropout_rate_shallow=config.dropout_rate_shallow
        )
        print("Created ResUNet3D model")
    
    model = model.to(device)
    
    # Wrap model with DataParallel to use multiple GPUs
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
        print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
    
    print(f"Model parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    
    return model


def create_optimizer(model: nn.Module, config: TrainConfig) -> torch.optim.Optimizer:
    """
    Create AdamW optimizer with weight decay
    
    Args:
        model: Model to optimize
        config: Training configuration
        
    Returns:
        Optimizer
    """
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=1e-5,
        betas=(0.9, 0.999)
    )
    print(f"Created AdamW optimizer with lr={config.learning_rate}")
    return optimizer


def create_scheduler(optimizer: torch.optim.Optimizer, config: TrainConfig, 
                     num_epochs: int) -> torch.optim.lr_scheduler._LRScheduler:
    """
    Create CosineAnnealingLR scheduler with warmup
    
    Note: Warmup is handled manually in the training loop
    
    Args:
        optimizer: Optimizer to schedule
        config: Training configuration
        num_epochs: Total number of epochs
        
    Returns:
        Learning rate scheduler
    """
    # CosineAnnealingLR - cycles the learning rate
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs,
        eta_min=config.learning_rate * 0.01  # Min LR = 1% of initial
    )
    print(f"Created CosineAnnealingLR scheduler (T_max={num_epochs})")
    return scheduler


def get_warmup_lr(epoch: int, warmup_epochs: int, base_lr: float, current_lr: float) -> float:
    """
    Calculate learning rate during warmup phase
    
    Linear warmup from base_lr * 0.1 to base_lr
    
    Args:
        epoch: Current epoch (0-indexed)
        warmup_epochs: Number of warmup epochs
        base_lr: Target learning rate after warmup
        current_lr: Current scheduled learning rate
        
    Returns:
        Learning rate to use
    """
    if epoch < warmup_epochs:
        # Linear warmup
        warmup_factor = (epoch + 1) / warmup_epochs
        return base_lr * warmup_factor * 0.1 + base_lr * (1 - 0.1) * warmup_factor
    return current_lr


def save_checkpoint(model: nn.Module, optimizer: torch.optim.Optimizer,
                   scheduler: torch.optim.lr_scheduler._LRScheduler,
                   epoch: int, metrics: Dict, config: TrainConfig,
                   filepath: str, is_best: bool = False):
    """
    Save training checkpoint
    
    Args:
        model: Model to save
        optimizer: Optimizer state
        scheduler: Scheduler state
        epoch: Current epoch
        metrics: Current metrics
        config: Training configuration
        filepath: Path to save checkpoint
        is_best: If True, this is the best model so far
    """
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'metrics': metrics,
        'config': {
            'num_classes': config.num_classes,
            'base_filters': config.base_filters,
            'depth': config.depth,
            'use_instance_norm': config.use_instance_norm,
            'use_learnable_fusion': config.use_learnable_fusion,
            'dropout_rate_deep': config.dropout_rate_deep,
            'dropout_rate_shallow': config.dropout_rate_shallow,
        },
        'is_best': is_best
    }
    
    torch.save(checkpoint, filepath)
    print(f"Checkpoint saved: {filepath}")


def load_checkpoint(filepath: str, model: nn.Module, 
                   optimizer: torch.optim.Optimizer = None,
                   scheduler: torch.optim.lr_scheduler._LRScheduler = None) -> Dict:
    """
    Load training checkpoint
    
    Args:
        filepath: Path to checkpoint
        model: Model to load weights into
        optimizer: Optional optimizer to load state into
        scheduler: Optional scheduler to load state into
        
    Returns:
        Checkpoint dictionary with epoch, metrics, etc.
    """
    checkpoint = torch.load(filepath, map_location='cpu')
    
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded model weights from epoch {checkpoint['epoch']}")
    
    if optimizer is not None and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        print("Loaded optimizer state")
    
    if scheduler is not None and 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        print("Loaded scheduler state")
    
    return checkpoint

print("Training utilities ready. Instantiate create_model/create_optimizer/create_scheduler directly before calling train_model with the real loaders built in Phase 3.")



### Cell 17: Enhanced Training Loop(Hrishikesh)


In [ ]:
# Cell 17: Enhanced Training Loop
def train_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer,
                scaler: GradScaler, config: TrainConfig, device: torch.device,
                grad_tracker: GradientNormTracker, ema_loss: EMALoss) -> Dict[str, any]:
    """
    Train for one epoch with gradient accumulation, AMP, and clipping
    
    Key features:
    1. Gradient accumulation (4 steps) for effective batch_size=4
    2. AMP (Automatic Mixed Precision) for memory efficiency
    3. Gradient clipping (max_norm=0.5) for stability
    4. Per-batch Dice monitoring for collapse detection
    5. EMA loss smoothing
    6. Gradient norm tracking
    
    Args:
        model: Model to train
        loader: Training data loader
        optimizer: Optimizer
        scaler: GradScaler for AMP
        config: Training configuration
        device: Device to train on
        grad_tracker: Gradient norm tracker
        ema_loss: EMA loss smoother
        
    Returns:
        Dictionary with training metrics:
        - loss: Average loss
        - ema_loss: EMA smoothed loss
        - per_class_dice: Per-class Dice scores
        - macro_tumor_dice: Macro tumor Dice
        - grad_norm_mean: Mean gradient norm
        - grad_norm_max: Max gradient norm
    """
    model.train()
    
    total_loss = 0.0
    batch_count = 0
    accumulation_steps = config.gradient_accumulation_steps
    
    # Accumulators for metrics
    all_dice_scores = defaultdict(list)
    grad_norms = []
    
    # Zero gradients at start
    optimizer.zero_grad()
    
    progress_bar = tqdm(loader, desc="Training", leave=False)
    
    for batch_idx, batch in enumerate(progress_bar):
        images = batch['image'].to(device)
        labels = batch['label'].to(device)
        
        # Forward pass with AMP
        with autocast():
            outputs = model(images)
            
            # Compute loss
            loss = combined_dice_focal_loss(
                outputs, labels,
                alpha=config.focal_alpha,
                gamma=config.focal_gamma,
                dice_weight=config.dice_weight,
                focal_weight=config.focal_weight,
                num_classes=config.num_classes
            )
            
            # Scale loss for gradient accumulation
            loss = loss / accumulation_steps
        
        # Backward pass with AMP scaling
        scaler.scale(loss).backward()
        
        # Compute Dice scores for this batch (for monitoring)
        with torch.no_grad():
            batch_dice = dice_score(outputs, labels, num_classes=config.num_classes)
            for c, score in batch_dice.items():
                all_dice_scores[c].append(score)
        
        # Update weights after accumulation_steps
        if (batch_idx + 1) % accumulation_steps == 0:
            # Unscale gradients before clipping
            scaler.unscale_(optimizer)
            
            # Compute gradient norm before clipping
            total_norm = 0.0
            for p in model.parameters():
                if p.grad is not None:
                    param_norm = p.grad.data.norm(2)
                    total_norm += param_norm.item() ** 2
            total_norm = total_norm ** 0.5
            grad_norms.append(total_norm)
            grad_tracker.update(total_norm)
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config.gradient_clip)
            
            # Optimizer step with scaler
            scaler.step(optimizer)
            scaler.update()
            
            # Zero gradients for next accumulation
            optimizer.zero_grad()
        
        # Track loss
        total_loss += loss.item() * accumulation_steps  # Unscale
        batch_count += 1
        
        # Update EMA loss
        smoothed_loss = ema_loss.update(loss.item() * accumulation_steps)
        
        # Update progress bar
        progress_bar.set_postfix({
            'loss': f'{smoothed_loss:.4f}',
            'dice': f'{np.mean([all_dice_scores[c][-1] for c in range(1, 5)]):.3f}'
        })
    
    # Handle remaining gradients (if batches not divisible by accumulation_steps)
    if batch_count % accumulation_steps != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config.gradient_clip)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
    
    # Compute average metrics
    avg_loss = total_loss / batch_count if batch_count > 0 else 0.0
    avg_dice = {c: np.mean(scores) for c, scores in all_dice_scores.items()}
    avg_macro_tumor_dice = macro_tumor_dice(avg_dice)
    
    # Gradient norm stats
    grad_norm_mean = np.mean(grad_norms) if grad_norms else 0.0
    grad_norm_max = np.max(grad_norms) if grad_norms else 0.0
    
    return {
        'loss': avg_loss,
        'ema_loss': ema_loss.ema_value,
        'per_class_dice': avg_dice,
        'macro_tumor_dice': avg_macro_tumor_dice,
        'grad_norm_mean': grad_norm_mean,
        'grad_norm_max': grad_norm_max
    }


print("train_epoch is wired to the real train_loader. Invoke it via train_model once Phase 3 dataloaders are instantiated.")


### Cell 18: Validation Loop(Hrishikesh)


In [ ]:
# Cell 18: Validation Loop
@torch.no_grad()
def validate_epoch(model: nn.Module, loader: DataLoader, config: TrainConfig,
                   device: torch.device, num_batches: int = 15) -> Dict[str, any]:
    """
    Validate model for one epoch with multi-batch evaluation
    
    Uses the eval_model function for consistent, deterministic evaluation.
    Also computes validation loss for monitoring.
    
    Args:
        model: Model to validate
        loader: Validation data loader
        config: Training configuration
        device: Device to run on
        num_batches: Number of batches to evaluate
        
    Returns:
        Dictionary with validation metrics:
        - loss: Average validation loss
        - per_class_dice: Per-class Dice scores
        - macro_tumor_dice: Macro tumor Dice (PRIMARY METRIC)
        - wt_tc_et: WT/TC/ET Dice scores
        - per_class_iou: Per-class IoU scores
    """
    model.eval()
    
    total_loss = 0.0
    batch_count = 0
    
    # Use eval_model for comprehensive metrics
    eval_results = eval_model(
        model=model,
        loader=loader,
        device=device,
        num_classes=config.num_classes,
        num_batches=num_batches,
        use_center_crop=config.use_center_crop_eval,
        crop_size=48
    )
    
    # Also compute validation loss
    for batch_idx, batch in enumerate(loader):
        if batch_idx >= num_batches:
            break
        
        images = batch['image'].to(device)
        labels = batch['label'].to(device)
        
        # Forward pass
        with autocast():
            outputs = model(images)
            
            loss = combined_dice_focal_loss(
                outputs, labels,
                alpha=config.focal_alpha,
                gamma=config.focal_gamma,
                dice_weight=config.dice_weight,
                focal_weight=config.focal_weight,
                num_classes=config.num_classes
            )
        
        total_loss += loss.item()
        batch_count += 1
    
    avg_loss = total_loss / batch_count if batch_count > 0 else 0.0
    
    # Combine loss with eval_results
    eval_results['loss'] = avg_loss
    
    return eval_results


print("validate_epoch consumes the real validation loader. It is executed automatically inside train_model during Phase 5 training.")


### Cell 19: Main Training Loop(Hrishikesh)


In [ ]:
# Cell 19: Main Training Loop
def train_model(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader,
                config: TrainConfig, device: torch.device) -> Dict[str, any]:
    """
    Main training loop with early stopping, checkpointing, and warmup
    
    Key features:
    1. Training history tracking
    2. Early stopping (patience=7, based on macro_tumor_dice)
    3. Best model checkpointing
    4. Milestone checkpointing (1/5th of total epochs)
    5. Learning rate warmup handling (3 epochs)
    6. Epoch summary printing
    
    Args:
        model: Model to train
        train_loader: Training data loader
        val_loader: Validation data loader
        config: Training configuration
        device: Device to train on
        
    Returns:
        Training history dictionary
    """
    print("="*60)
    print("STARTING TRAINING")
    print("="*60)
    print(f"Model: {model.__class__.__name__}")
    print(f"Epochs: {config.epochs}")
    print(f"Batch size: {config.batch_size}")
    print(f"Learning rate: {config.learning_rate}")
    print(f"Gradient accumulation: {config.gradient_accumulation_steps}")
    print("="*60)
    
    # Create training components
    optimizer = create_optimizer(model, config)
    scheduler = create_scheduler(optimizer, config, config.epochs)
    scaler = GradScaler()
    grad_tracker = GradientNormTracker(window_size=100)
    ema_loss = EMALoss(alpha=0.1)
    
    # Training history
    history = {
        'train_loss': [],
        'val_loss': [],
        'train_dice': [],
        'val_dice': [],
        'macro_tumor_dice': [],
        'wt_tc_et': [],
        'grad_norms': [],
        'learning_rates': []
    }
    
    # Early stopping
    best_metric = 0.0
    best_epoch = 0
    patience = 7
    patience_counter = 0
    warmup_epochs = 3
    milestone_interval = max(1, math.ceil(max(1, config.epochs) / 5))
    milestone_epochs = set(range(milestone_interval, config.epochs + 1, milestone_interval))
    print(f"Milestone checkpoints planned at epochs: {sorted(milestone_epochs)}")
    
    # Training loop
    for epoch in range(config.epochs):
        print(f"\n{'='*60}")
        print(f"Epoch {epoch + 1}/{config.epochs}")
        print(f"{'='*60}")
        
        # Apply warmup LR
        current_lr = optimizer.param_groups[0]['lr']
        if epoch < warmup_epochs:
            warmup_lr = get_warmup_lr(epoch, warmup_epochs, config.learning_rate, current_lr)
            for param_group in optimizer.param_groups:
                param_group['lr'] = warmup_lr
            print(f"Warmup LR: {warmup_lr:.6f}")
        else:
            print(f"Learning rate: {current_lr:.6f}")
        
        history['learning_rates'].append(optimizer.param_groups[0]['lr'])
        
        # Training epoch
        train_metrics = train_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            scaler=scaler,
            config=config,
            device=device,
            grad_tracker=grad_tracker,
            ema_loss=ema_loss
        )
        
        # Validation epoch
        val_metrics = validate_epoch(
            model=model,
            loader=val_loader,
            config=config,
            device=device,
            num_batches=config.eval_batches
        )
        
        # Update scheduler (after warmup)
        if epoch >= warmup_epochs:
            scheduler.step()
        
        # Record history
        history['train_loss'].append(train_metrics['loss'])
        history['val_loss'].append(val_metrics['loss'])
        history['train_dice'].append(train_metrics['per_class_dice'])
        history['val_dice'].append(val_metrics['per_class_dice'])
        history['macro_tumor_dice'].append(val_metrics['macro_tumor_dice'])
        history['wt_tc_et'].append(val_metrics['wt_tc_et'])
        history['grad_norms'].append({
            'mean': train_metrics['grad_norm_mean'],
            'max': train_metrics['grad_norm_max']
        })
        
        # Print epoch summary
        print(f"\n--- Epoch {epoch + 1} Summary ---")
        print(f"Train Loss: {train_metrics['loss']:.4f} | Val Loss: {val_metrics['loss']:.4f}")
        print(f"Train Macro Dice: {train_metrics['macro_tumor_dice']:.4f} | Val Macro Dice: {val_metrics['macro_tumor_dice']:.4f}")
        print(f"WT: {val_metrics['wt_tc_et']['WT']:.4f} | TC: {val_metrics['wt_tc_et']['TC']:.4f} | ET: {val_metrics['wt_tc_et']['ET']:.4f}")
        print(f"Grad Norm: mean={train_metrics['grad_norm_mean']:.4f}, max={train_metrics['grad_norm_max']:.4f}")
        
        # Print per-class validation Dice
        print("Val Dice per class:", end=" ")
        for c, score in val_metrics['per_class_dice'].items():
            print(f"C{c}={score:.3f}", end=" ")
        print()
        
        # Print fusion weights if applicable
        if config.use_learnable_fusion and hasattr(model, 'print_fusion_weights'):
            model.print_fusion_weights()
        
        # Early stopping check (based on macro_tumor_dice)
        current_metric = val_metrics['macro_tumor_dice']
        
        if current_metric > best_metric:
            best_metric = current_metric
            best_epoch = epoch + 1
            patience_counter = 0
            
            # Save best model
            best_path = os.path.join(config.checkpoint_dir, 'best_model.pt')
            save_checkpoint(
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                epoch=epoch,
                metrics=val_metrics,
                config=config,
                filepath=best_path,
                is_best=True
            )
            print(f"*** New best model! Macro Tumor Dice: {best_metric:.4f} ***")
        else:
            patience_counter += 1
            print(f"No improvement. Patience: {patience_counter}/{patience}")
        
        # Milestone checkpointing (1/5th intervals)
        current_epoch = epoch + 1
        if current_epoch in milestone_epochs or current_epoch == config.epochs:
            milestone_path = os.path.join(
                config.checkpoint_dir,
                f'checkpoint_milestone_{current_epoch}.pt'
            )
            save_checkpoint(
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                epoch=epoch,
                metrics=val_metrics,
                config=config,
                filepath=milestone_path,
                is_best=False
            )
            print(f"Saved milestone checkpoint at epoch {current_epoch}")
        
        # Early stopping
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch + 1}")
            break
        
        # Clear GPU cache
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    # Training complete
    print("\n" + "="*60)
    print("TRAINING COMPLETE")
    print("="*60)
    print(f"Best Macro Tumor Dice: {best_metric:.4f} at epoch {best_epoch}")
    print(f"Best model saved to: {os.path.join(config.checkpoint_dir, 'best_model.pt')}")
    
    # Save final model
    final_path = os.path.join(config.checkpoint_dir, 'final_model.pt')
    save_checkpoint(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        epoch=epoch,
        metrics=val_metrics,
        config=config,
        filepath=final_path,
        is_best=False
    )
    
    history['best_metric'] = best_metric
    history['best_epoch'] = best_epoch
    
    return history

print("Run train_model(model, train_loader, val_loader, config, device) once the loaders from Phase 3 are available. No synthetic loaders are used for dry runs.")



### Cell 20b: Real Data Training Orchestration(Hrishikesh)
Once `train_loader` and `val_loader` are instantiated in Phase 3, toggle the switch below to launch a full training run on the actual BraTS data. The flag defaults to `False` so simply importing the notebook does not start a multi-hour job.


In [ ]:
RUN_FULL_TRAINING = False  # Flip to True to launch the real training job

if RUN_FULL_TRAINING:
    if 'train_loader' not in locals() or 'val_loader' not in locals():
        raise RuntimeError("Phase 3 dataloaders not found. Re-run the data preparation cells before training.")
    if len(train_loader) == 0 or len(val_loader) == 0:
        raise RuntimeError("Dataloaders are empty. Confirm that the dataset paths are correct and cases were discovered.")

    print("Launching end-to-end training on the full BraTS training split...")
    trained_model = create_model(config, device)
    training_history = train_model(
        model=trained_model,
        train_loader=train_loader,
        val_loader=val_loader,
        config=config,
        device=device
    )
else:
    print("Set RUN_FULL_TRAINING=True to train on the actual BraTS loaders prepared in Phase 3.")



### Cell 20: Training Visualization(Hrishikesh)


In [ ]:
# Cell 20: Training Visualization
def plot_training_curves(history: Dict[str, any], output_dir: str = None, 
                         show_plot: bool = True) -> None:
    """
    Plot comprehensive training curves
    
    Generates a 2x3 grid of plots:
    1. Loss curves (train/val)
    2. Macro tumor Dice curve
    3. Per-class Dice curves (validation)
    4. WT/TC/ET Dice curves
    5. Gradient norm plot
    6. Learning rate schedule plot
    
    Args:
        history: Training history dictionary
        output_dir: Directory to save plots (if provided)
        show_plot: Whether to display the plot
    """
    # Create figure with 2x3 subplots
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle('Training Progress', fontsize=14, fontweight='bold')
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Plot 1: Loss curves
    ax1 = axes[0, 0]
    ax1.plot(epochs, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
    ax1.plot(epochs, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training & Validation Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Macro Tumor Dice
    ax2 = axes[0, 1]
    ax2.plot(epochs, history['macro_tumor_dice'], 'g-', linewidth=2, marker='o', markersize=4)
    ax2.axhline(y=max(history['macro_tumor_dice']), color='g', linestyle='--', alpha=0.5, 
                label=f'Best: {max(history["macro_tumor_dice"]):.4f}')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Macro Tumor Dice')
    ax2.set_title('Macro Tumor Dice (Primary Metric)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim([0, 1])
    
    # Plot 3: Per-class Dice (Validation)
    ax3 = axes[0, 2]
    class_names = {0: 'BG', 1: 'Necrosis', 2: 'Edema', 3: 'Non-Enh', 4: 'Enhancing'}
    colors = ['gray', 'red', 'green', 'orange', 'blue']
    
    for c in range(5):
        class_dice = [d.get(c, 0) for d in history['val_dice']]
        ax3.plot(epochs, class_dice, label=class_names[c], color=colors[c], linewidth=1.5)
    
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('Dice Score')
    ax3.set_title('Per-class Validation Dice')
    ax3.legend(loc='lower right', fontsize=8)
    ax3.grid(True, alpha=0.3)
    ax3.set_ylim([0, 1])
    
    # Plot 4: WT/TC/ET Dice
    ax4 = axes[1, 0]
    wt_scores = [w['WT'] for w in history['wt_tc_et']]
    tc_scores = [w['TC'] for w in history['wt_tc_et']]
    et_scores = [w['ET'] for w in history['wt_tc_et']]
    
    ax4.plot(epochs, wt_scores, 'b-', label='Whole Tumor (WT)', linewidth=2)
    ax4.plot(epochs, tc_scores, 'r-', label='Tumor Core (TC)', linewidth=2)
    ax4.plot(epochs, et_scores, 'g-', label='Enhancing Tumor (ET)', linewidth=2)
    ax4.set_xlabel('Epoch')
    ax4.set_ylabel('Dice Score')
    ax4.set_title('BraTS Region Dice (WT/TC/ET)')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    ax4.set_ylim([0, 1])
    
    # Plot 5: Gradient Norms
    ax5 = axes[1, 1]
    grad_means = [g['mean'] for g in history['grad_norms']]
    grad_maxs = [g['max'] for g in history['grad_norms']]
    
    ax5.plot(epochs, grad_means, 'b-', label='Mean Grad Norm', linewidth=2)
    ax5.fill_between(epochs, grad_means, grad_maxs, alpha=0.3, color='blue', label='Max Grad Norm')
    ax5.set_xlabel('Epoch')
    ax5.set_ylabel('Gradient Norm')
    ax5.set_title('Gradient Norm (Stability Indicator)')
    ax5.legend()
    ax5.grid(True, alpha=0.3)
    
    # Plot 6: Learning Rate Schedule
    ax6 = axes[1, 2]
    ax6.plot(epochs, history['learning_rates'], 'purple', linewidth=2)
    ax6.set_xlabel('Epoch')
    ax6.set_ylabel('Learning Rate')
    ax6.set_title('Learning Rate Schedule')
    ax6.grid(True, alpha=0.3)
    ax6.ticklabel_format(axis='y', style='scientific', scilimits=(0,0))
    
    plt.tight_layout()
    
    # Save plot
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        save_path = os.path.join(output_dir, 'training_curves.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Training curves saved to: {save_path}")
    
    if show_plot:
        plt.show()
    else:
        plt.close()


def plot_class_comparison(history: Dict[str, any], output_dir: str = None,
                         show_plot: bool = True) -> None:
    """
    Plot detailed per-class comparison between training and validation
    
    Args:
        history: Training history dictionary
        output_dir: Directory to save plot
        show_plot: Whether to display the plot
    """
    class_names = {0: 'Background', 1: 'Necrosis', 2: 'Edema', 3: 'Non-Enhancing', 4: 'Enhancing'}
    
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    fig.suptitle('Per-Class Training vs Validation Dice', fontsize=12, fontweight='bold')
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    for c in range(5):
        ax = axes[c]
        
        train_dice = [d.get(c, 0) for d in history['train_dice']]
        val_dice = [d.get(c, 0) for d in history['val_dice']]
        
        ax.plot(epochs, train_dice, 'b-', label='Train', linewidth=1.5)
        ax.plot(epochs, val_dice, 'r-', label='Val', linewidth=1.5)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Dice Score')
        ax.set_title(f'{class_names[c]}')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.set_ylim([0, 1])
    
    plt.tight_layout()
    
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        save_path = os.path.join(output_dir, 'class_comparison.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Class comparison saved to: {save_path}")
    
    if show_plot:
        plt.show()
    else:
        plt.close()

print("Call plot_training_curves/plot_class_comparison with the real `training_history` returned by train_model once a full run completes.")



## Phase 6: Testing and Final Evaluation(Hrishikesh)

### Cell 21: Load Best Model(Hrishikesh)


In [ ]:
# Cell 21: Load Best Model
def load_best_model(checkpoint_path: str, config: TrainConfig, device: torch.device) -> Tuple[nn.Module, Dict]:
    """
    Load the best model from checkpoint for evaluation
    
    Args:
        checkpoint_path: Path to checkpoint file
        config: Training configuration
        device: Device to load model on
        
    Returns:
        Tuple of (model, checkpoint_info)
    """
    print(f"Loading checkpoint from: {checkpoint_path}")
    
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    ckpt_config = checkpoint.get('config', {})
    if ckpt_config:
        print(f"Checkpoint config: {ckpt_config}")
    
    # Determine fusion setting from checkpoint (fallback to current config)
    effective_use_learnable = ckpt_config.get('use_learnable_fusion', config.use_learnable_fusion)
    config_for_model = copy.deepcopy(config)
    config_for_model.use_learnable_fusion = effective_use_learnable
    
    # Create model
    model = create_model(config_for_model, device)
    
    # Load weights
    # Load weights (handle DataParallel state dict)
    state_dict = checkpoint['model_state_dict']
    # Remove 'module.' prefix if present (for DataParallel compatibility)
    if any(k.startswith('module.') for k in state_dict.keys()):
        # Checkpoint was saved with DataParallel
        if isinstance(model, nn.DataParallel):
            model.load_state_dict(state_dict)
        else:
            # Model is not DataParallel but checkpoint is, remove prefix
            new_state_dict = {k[7:] if k.startswith('module.') else k: v for k, v in state_dict.items()}
            model.load_state_dict(new_state_dict)
    else:
        # Checkpoint was saved without DataParallel
        if isinstance(model, nn.DataParallel):
            # Model is DataParallel but checkpoint is not, add prefix
            new_state_dict = {'module.' + k: v for k, v in state_dict.items()}
            model.load_state_dict(new_state_dict)
        else:
            model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()
    
    # Extract checkpoint info
    checkpoint_info = {
        'epoch': checkpoint.get('epoch', 'unknown'),
        'metrics': checkpoint.get('metrics', {}),
        'is_best': checkpoint.get('is_best', False),
        'use_learnable_fusion': effective_use_learnable
    }
    
    print(f"Model loaded successfully!")
    print(f"  Trained for: {checkpoint_info['epoch']} epochs")
    if checkpoint_info['metrics']:
        metrics = checkpoint_info['metrics']
        if 'macro_tumor_dice' in metrics:
            print(f"  Best Macro Tumor Dice: {metrics['macro_tumor_dice']:.4f}")
    
    return model, checkpoint_info

print("load_best_model should be called with the checkpoints produced by train_model. Once training finishes, pass the best-model path instead of relying on any synthetic checkpoints.")



### Cell 22: Test Set Evaluation(Hrishikesh)


In [ ]:
# Cell 22: Test Set Evaluation
# Literature baselines for comparison
LITERATURE_BASELINES = {
    'Standard U-Net': {'dice_range': (0.61, 0.889), 'source': 'Various papers'},
    'Multimodal Fusion U-Net': {'dice_range': (0.75, 0.75), 'source': 'Project report'},
    'Res-UNet': {'dice_range': (0.86, 0.9062), 'source': 'Various papers'},
    'Swin-UNet-EPA': {'dice_range': (0.915, 0.915), 'source': 'State-of-the-art'},
}

def evaluate_test_set(model: nn.Module, test_loader: DataLoader, config: TrainConfig,
                      device: torch.device, num_batches: int = None) -> Dict[str, any]:
    """
    Comprehensive test set evaluation with literature comparison
    
    Args:
        model: Model to evaluate
        test_loader: Test data loader
        config: Training configuration
        device: Device to run on
        num_batches: Number of batches to evaluate (None = all)
        
    Returns:
        Dictionary with all evaluation metrics and comparison
    """
    print("="*60)
    print("TEST SET EVALUATION")
    print("="*60)
    
    model.eval()
    
    # Use all batches if not specified
    if num_batches is None:
        num_batches = len(test_loader)
    
    # Run evaluation
    results = eval_model(
        model=model,
        loader=test_loader,
        device=device,
        num_classes=config.num_classes,
        num_batches=num_batches,
        use_center_crop=False,  # Use full patches for final evaluation
        crop_size=48
    )
    
    # Compute additional statistics
    results['mean_tumor_dice'] = np.mean([
        results['per_class_dice'].get(c, 0) for c in range(1, 5)
    ])
    
    # Print comprehensive results
    print_eval_results(results)
    
    # Compare with literature
    print("\n" + "="*60)
    print("COMPARISON WITH LITERATURE")
    print("="*60)
    
    our_dice = results['macro_tumor_dice']
    
    for method, baseline in LITERATURE_BASELINES.items():
        dice_range = baseline['dice_range']
        avg_baseline = (dice_range[0] + dice_range[1]) / 2
        diff = our_dice - avg_baseline
        status = "+" if diff > 0 else ""
        
        print(f"\n{method}:")
        print(f"  Literature Dice: {dice_range[0]*100:.1f}% - {dice_range[1]*100:.1f}%")
        print(f"  Our Dice: {our_dice*100:.2f}%")
        print(f"  Difference: {status}{diff*100:.2f}%")
    
    # Store comparison in results
    results['literature_comparison'] = {
        method: {
            'baseline_range': baseline['dice_range'],
            'our_score': our_dice,
            'improvement': our_dice - (baseline['dice_range'][0] + baseline['dice_range'][1]) / 2
        }
        for method, baseline in LITERATURE_BASELINES.items()
    }
    
    return results


def print_test_summary(results: Dict[str, any]) -> None:
    """Print a formatted test summary"""
    print("\n" + "="*60)
    print("FINAL TEST RESULTS SUMMARY")
    print("="*60)
    
    print(f"\n{'Metric':<30} {'Score':>10}")
    print("-"*42)
    
    # Primary metric
    print(f"{'Macro Tumor Dice (PRIMARY)':<30} {results['macro_tumor_dice']*100:>9.2f}%")
    
    # BraTS metrics
    print(f"\n{'BraTS Challenge Metrics:':<30}")
    print(f"{'  Whole Tumor (WT)':<30} {results['wt_tc_et']['WT']*100:>9.2f}%")
    print(f"{'  Tumor Core (TC)':<30} {results['wt_tc_et']['TC']*100:>9.2f}%")
    print(f"{'  Enhancing Tumor (ET)':<30} {results['wt_tc_et']['ET']*100:>9.2f}%")
    
    # Per-class Dice
    class_names = {0: 'Background', 1: 'Necrosis', 2: 'Edema', 3: 'Non-Enhancing', 4: 'Enhancing'}
    print(f"\n{'Per-Class Dice Scores:':<30}")
    for c in range(5):
        score = results['per_class_dice'].get(c, 0)
        print(f"{'  ' + class_names[c]:<30} {score*100:>9.2f}%")
    
    print("-"*42)


print("evaluate_test_set expects the validation loader produced in Phase 3. After training, call it with (model, val_loader, config, device) to generate real metrics—no dummy loaders are provided.")


### Cell 23: Qualitative Visualization(Nihal)


In [ ]:
# Cell 23: Qualitative Visualization
# Color mapping for tumor classes
TUMOR_COLORS = {
    0: [0, 0, 0, 0],        # Background: transparent
    1: [255, 0, 0, 180],    # Necrosis: Red
    2: [0, 255, 0, 180],    # Edema: Green
    3: [255, 255, 0, 180],  # Non-enhancing: Yellow
    4: [0, 0, 255, 180]     # Enhancing: Blue
}

def create_overlay(image_slice: np.ndarray, label_slice: np.ndarray, 
                   alpha: float = 0.4) -> np.ndarray:
    """
    Create RGB overlay of segmentation on grayscale image
    
    Args:
        image_slice: 2D grayscale image slice
        label_slice: 2D label slice with class indices
        alpha: Overlay transparency
        
    Returns:
        RGB image with overlay
    """
    # Normalize image to 0-255
    img_norm = ((image_slice - image_slice.min()) / 
                (image_slice.max() - image_slice.min() + 1e-8) * 255).astype(np.uint8)
    
    # Create RGB image from grayscale
    rgb_img = np.stack([img_norm, img_norm, img_norm], axis=-1)
    
    # Create overlay
    overlay = rgb_img.copy().astype(float)
    
    for class_id, color in TUMOR_COLORS.items():
        if class_id == 0:  # Skip background
            continue
        mask = (label_slice == class_id)
        if mask.any():
            for c in range(3):
                overlay[..., c] = np.where(mask, 
                    (1 - alpha) * rgb_img[..., c] + alpha * color[c], 
                    overlay[..., c])
    
    return overlay.astype(np.uint8)


@torch.no_grad()
def visualize_predictions(model: nn.Module, loader: DataLoader, device: torch.device,
                         num_samples: int = 3, output_dir: str = None,
                         show_plot: bool = True) -> None:
    """
    Visualize segmentation predictions with ground truth comparison
    
    For each sample, shows:
    - Input image (middle slice)
    - Ground truth overlay
    - Prediction overlay
    - Difference map
    
    Args:
        model: Model to use for prediction
        loader: Data loader
        device: Device to run on
        num_samples: Number of samples to visualize
        output_dir: Directory to save visualizations
        show_plot: Whether to display plots
    """
    model.eval()
    
    samples_shown = 0
    
    for batch in loader:
        if samples_shown >= num_samples:
            break
        
        images = batch['image'].to(device)
        labels = batch['label'].to(device)
        case_ids = batch['case_id']
        
        # Get predictions
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        
        # Process each sample in batch
        for i in range(images.shape[0]):
            if samples_shown >= num_samples:
                break
            
            image = images[i].cpu().numpy()
            label = labels[i].cpu().numpy()
            pred = preds[i].cpu().numpy()
            case_id = case_ids[i]
            
            # Get middle slice indices
            d, h, w = label.shape
            mid_d = d // 2
            mid_h = h // 2
            mid_w = w // 2
            
            # Create figure with 3 views x 4 columns
            fig, axes = plt.subplots(3, 4, figsize=(16, 12))
            fig.suptitle(f'Sample: {case_id}', fontsize=14, fontweight='bold')
            
            # Column titles
            col_titles = ['Input Image', 'Ground Truth', 'Prediction', 'Difference']
            
            # Row titles (views)
            row_titles = ['Axial (z)', 'Coronal (y)', 'Sagittal (x)']
            
            # Get slices for each view
            views = [
                # (image_slice, label_slice, pred_slice)
                (image[0, mid_d, :, :], label[mid_d, :, :], pred[mid_d, :, :]),  # Axial
                (image[0, :, mid_h, :], label[:, mid_h, :], pred[:, mid_h, :]),  # Coronal
                (image[0, :, :, mid_w], label[:, :, mid_w], pred[:, :, mid_w])   # Sagittal
            ]
            
            for row, (img_slice, lbl_slice, prd_slice) in enumerate(views):
                # Column 0: Input image
                axes[row, 0].imshow(img_slice, cmap='gray')
                axes[row, 0].set_title(col_titles[0] if row == 0 else '')
                axes[row, 0].set_ylabel(row_titles[row])
                axes[row, 0].axis('off')
                
                # Column 1: Ground truth overlay
                gt_overlay = create_overlay(img_slice, lbl_slice)
                axes[row, 1].imshow(gt_overlay)
                axes[row, 1].set_title(col_titles[1] if row == 0 else '')
                axes[row, 1].axis('off')
                
                # Column 2: Prediction overlay
                pred_overlay = create_overlay(img_slice, prd_slice)
                axes[row, 2].imshow(pred_overlay)
                axes[row, 2].set_title(col_titles[2] if row == 0 else '')
                axes[row, 2].axis('off')
                
                # Column 3: Difference (errors)
                diff = (lbl_slice != prd_slice).astype(float)
                axes[row, 3].imshow(img_slice, cmap='gray', alpha=0.5)
                axes[row, 3].imshow(diff, cmap='Reds', alpha=0.5)
                axes[row, 3].set_title(col_titles[3] if row == 0 else '')
                axes[row, 3].axis('off')
            
            # Add legend
            legend_elements = [
                plt.Rectangle((0,0), 1, 1, facecolor='red', alpha=0.7, label='Necrosis'),
                plt.Rectangle((0,0), 1, 1, facecolor='green', alpha=0.7, label='Edema'),
                plt.Rectangle((0,0), 1, 1, facecolor='yellow', alpha=0.7, label='Non-Enhancing'),
                plt.Rectangle((0,0), 1, 1, facecolor='blue', alpha=0.7, label='Enhancing')
            ]
            fig.legend(handles=legend_elements, loc='lower center', ncol=4, 
                      bbox_to_anchor=(0.5, -0.02))
            
            plt.tight_layout()
            
            if output_dir:
                os.makedirs(output_dir, exist_ok=True)
                save_path = os.path.join(output_dir, f'prediction_{case_id}.png')
                plt.savefig(save_path, dpi=150, bbox_inches='tight')
                print(f"Saved: {save_path}")
            
            if show_plot:
                plt.show()
            else:
                plt.close()
            
            samples_shown += 1
    
    print(f"\nVisualized {samples_shown} samples")


# Test visualization
print("visualize_predictions should be called with the trained model and the real validation loader to inspect predictions. No dummy loaders are invoked in this phase.")


### Cell 24: Performance Summary and Export(Nihal)


In [ ]:
# Cell 24: Performance Summary and Export
import json
import pandas as pd

def create_performance_summary(test_results: Dict[str, any], history: Dict[str, any],
                               output_dir: str, model_name: str = "ResUNet3D") -> pd.DataFrame:
    """
    Create comprehensive performance summary and export to files
    
    Generates:
    1. Summary DataFrame with all metrics
    2. Comparison with literature baselines
    3. Export to CSV and JSON
    
    Args:
        test_results: Test evaluation results
        history: Training history
        output_dir: Directory to save outputs
        model_name: Name of the model for labeling
        
    Returns:
        Summary DataFrame
    """
    os.makedirs(output_dir, exist_ok=True)
    
    print("="*60)
    print("PERFORMANCE SUMMARY")
    print("="*60)
    
    # Create summary data
    summary_data = []
    
    # Primary metric
    summary_data.append({
        'Category': 'Primary',
        'Metric': 'Macro Tumor Dice',
        'Value': test_results['macro_tumor_dice'],
        'Notes': 'Average of tumor classes 1-4'
    })
    
    # BraTS metrics
    for region, score in test_results['wt_tc_et'].items():
        region_full = {'WT': 'Whole Tumor', 'TC': 'Tumor Core', 'ET': 'Enhancing Tumor'}[region]
        summary_data.append({
            'Category': 'BraTS',
            'Metric': region_full,
            'Value': score,
            'Notes': f'Standard BraTS challenge metric'
        })
    
    # Per-class Dice
    class_names = {0: 'Background', 1: 'Necrosis', 2: 'Edema', 3: 'Non-Enhancing', 4: 'Enhancing'}
    for c, score in test_results['per_class_dice'].items():
        summary_data.append({
            'Category': 'Per-Class Dice',
            'Metric': class_names[c],
            'Value': score,
            'Notes': f'Class {c}'
        })
    
    # Per-class IoU
    for c, score in test_results['per_class_iou'].items():
        summary_data.append({
            'Category': 'Per-Class IoU',
            'Metric': class_names[c],
            'Value': score,
            'Notes': f'Class {c}'
        })
    
    # Create DataFrame
    df = pd.DataFrame(summary_data)
    
    # Print summary table
    print("\n" + df.to_string(index=False))
    
    # Literature comparison table
    print("\n" + "="*60)
    print("LITERATURE COMPARISON")
    print("="*60)
    
    comparison_data = []
    our_dice = test_results['macro_tumor_dice']
    
    for method, baseline in LITERATURE_BASELINES.items():
        dice_range = baseline['dice_range']
        avg_baseline = (dice_range[0] + dice_range[1]) / 2
        diff = our_dice - avg_baseline
        
        comparison_data.append({
            'Method': method,
            'Baseline Dice': f"{dice_range[0]*100:.1f}%-{dice_range[1]*100:.1f}%",
            'Our Dice': f"{our_dice*100:.2f}%",
            'Difference': f"{diff*100:+.2f}%"
        })
    
    # Add our model
    comparison_data.append({
        'Method': f'{model_name} (Ours)',
        'Baseline Dice': '-',
        'Our Dice': f"{our_dice*100:.2f}%",
        'Difference': '-'
    })
    
    comparison_df = pd.DataFrame(comparison_data)
    print("\n" + comparison_df.to_string(index=False))
    
    # Training summary
    if history and 'best_metric' in history:
        print("\n" + "="*60)
        print("TRAINING SUMMARY")
        print("="*60)
        print(f"Best Epoch: {history['best_epoch']}")
        print(f"Best Validation Macro Dice: {history['best_metric']:.4f}")
        print(f"Final Training Loss: {history['train_loss'][-1]:.4f}")
        print(f"Final Validation Loss: {history['val_loss'][-1]:.4f}")
    
    # Export to files
    # CSV
    csv_path = os.path.join(output_dir, 'performance_summary.csv')
    df.to_csv(csv_path, index=False)
    print(f"\nSaved CSV: {csv_path}")
    
    comparison_csv = os.path.join(output_dir, 'literature_comparison.csv')
    comparison_df.to_csv(comparison_csv, index=False)
    print(f"Saved comparison CSV: {comparison_csv}")
    
    # JSON
    json_data = {
        'model_name': model_name,
        'test_results': {
            'macro_tumor_dice': test_results['macro_tumor_dice'],
            'wt_tc_et': test_results['wt_tc_et'],
            'per_class_dice': {str(k): v for k, v in test_results['per_class_dice'].items()},
            'per_class_iou': {str(k): v for k, v in test_results['per_class_iou'].items()}
        },
        'training_info': {
            'best_epoch': history.get('best_epoch', None),
            'best_metric': history.get('best_metric', None),
            'total_epochs': len(history['train_loss']) if history else 0
        } if history else None,
        'literature_comparison': {
            method: {
                'baseline_range': list(baseline['dice_range']),
                'our_score': our_dice,
                'improvement': our_dice - (baseline['dice_range'][0] + baseline['dice_range'][1]) / 2
            }
            for method, baseline in LITERATURE_BASELINES.items()
        }
    }
    
    json_path = os.path.join(output_dir, 'results.json')
    with open(json_path, 'w') as f:
        json.dump(json_data, f, indent=2)
    print(f"Saved JSON: {json_path}")
    
    return df


def run_full_evaluation_pipeline(checkpoint_path: str, test_loader: DataLoader,
                                 config: TrainConfig, device: torch.device,
                                 history: Dict = None, output_dir: str = './results') -> Dict:
    """
    Run the complete evaluation pipeline
    
    Args:
        checkpoint_path: Path to model checkpoint
        test_loader: Test data loader
        config: Training configuration
        device: Device to run on
        history: Training history (if available)
        output_dir: Output directory for results
        
    Returns:
        Complete results dictionary
    """
    print("\n" + "="*70)
    print("FULL EVALUATION PIPELINE")
    print("="*70)
    
    # Step 1: Load model
    print("\n[Step 1/4] Loading model...")
    model, checkpoint_info = load_best_model(
        checkpoint_path=checkpoint_path,
        config=config,
        device=device
    )
    
    # Step 2: Evaluate on test set
    print("\n[Step 2/4] Evaluating on test set...")
    test_results = evaluate_test_set(
        model=model,
        test_loader=test_loader,
        config=config,
        device=device
    )
    
    # Step 3: Generate visualizations
    print("\n[Step 3/4] Generating visualizations...")
    visualize_predictions(
        model=model,
        loader=test_loader,
        device=device,
        num_samples=3,
        output_dir=output_dir,
        show_plot=False
    )
    
    # Step 4: Create summary
    print("\n[Step 4/4] Creating performance summary...")
    model_name = "LearnableFusionResUNet3D" if config.use_learnable_fusion else "ResUNet3D"
    summary_df = create_performance_summary(
        test_results=test_results,
        history=history,
        output_dir=output_dir,
        model_name=model_name
    )
    
    print("\n" + "="*70)
    print("EVALUATION COMPLETE")
    print("="*70)
    print(f"Results saved to: {output_dir}")
    
    return {
        'test_results': test_results,
        'checkpoint_info': checkpoint_info,
        'summary_df': summary_df
    }


# Run summary with available data
print("Creating performance summary from the most recent validation evaluation:")

if 'validation_eval' in locals():
    summary_df = validation_eval['summary_df']
    print("Summary already generated via run_full_evaluation_pipeline.")
elif 'test_results' in locals():
    history_to_use = training_history if 'training_history' in locals() else None
    summary_df = create_performance_summary(
        test_results=test_results,
        history=history_to_use,
        output_dir=config.output_dir,
        model_name='ResUNet3D'
    )
else:
    print("No evaluation results available. Set RUN_VALIDATION_EVAL=True after training to populate metrics.")

print("\n" + "="*60)
print("IMPLEMENTATION COMPLETE")
print("="*60)
print("""
Phase 5-6 Implementation Summary:

Phase 5: Training Pipeline
- Cell 16: Training setup (model, optimizer, scheduler, scaler)
- Cell 17: train_epoch() with gradient accumulation, AMP, clipping
- Cell 18: validate_epoch() with multi-batch evaluation
- Cell 19: Main training loop with early stopping and checkpointing
- Cell 20: Training visualization (loss, dice, grad norms, LR)

Phase 6: Testing and Final Evaluation
- Cell 21: Load best model from checkpoint
- Cell 22: Test set evaluation with literature comparison
- Cell 23: Qualitative visualization of predictions
- Cell 24: Performance summary and export

To run full training on real data:
1. Ensure BraTS dataset is available
2. Create train/val/test splits
3. Call train_model() with real dataloaders
4. Call run_full_evaluation_pipeline() after training

Expected performance:
- Macro Tumor Dice: >85% (competitive with Res-UNet literature)
- With learnable fusion: potential +5-10% improvement
""")


## Phase 7: Ablation Studies(Nihal)
### Cell 25: Fusion Ablation Study(Nihal)


In [ ]:
# Cell 25: Fusion Ablation Study
class MeanFusionResUNet3D(nn.Module):
    def __init__(self, backbone: nn.Module):
        super().__init__()
        self.backbone = backbone

    def forward(self, x):
        # fuse 4→1
        x = x.mean(dim=1, keepdim=True)
        return self.backbone(x)
class StaticWeightedFusionResUNet3D(nn.Module):
    """
    ResUNet3D with static weighted fusion
    
    Uses fixed weights for fusing 4 MRI modalities before the backbone.
    Weights: [0.15, 0.35, 0.25, 0.25] for [T1, T1ce, T2, FLAIR]
    (T1ce typically most informative for tumor detection)
    """
    def __init__(self, num_classes: int = 5, base_filters: int = 16,
                 depth: int = 4, use_instance_norm: bool = True,
                 dropout_rate_deep: float = 0.25, dropout_rate_shallow: float = 0.0,
                 fusion_weights: List[float] = None):
        super(StaticWeightedFusionResUNet3D, self).__init__()
        
        self.num_modalities = 4
        
        # Static fusion weights (can be customized)
        if fusion_weights is None:
            fusion_weights = [0.15, 0.35, 0.25, 0.25]  # T1, T1ce, T2, FLAIR
        
        self.register_buffer('fusion_weights', torch.tensor(fusion_weights))
        
        # Backbone
        self.backbone = ResUNet3D(
            in_channels=1,
            num_classes=num_classes,
            base_filters=base_filters,
            depth=depth,
            use_instance_norm=use_instance_norm,
            dropout_rate_deep=dropout_rate_deep,
            dropout_rate_shallow=dropout_rate_shallow
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass with static weighted fusion
        
        Args:
            x: Input tensor (B, 4, D, H, W) with 4 modalities
            
        Returns:
            Output logits (B, num_classes, D, H, W)
        """
        # Apply weighted fusion
        weights = self.fusion_weights.view(1, self.num_modalities, 1, 1, 1)
        fused = (x * weights).sum(dim=1, keepdim=True)  # (B, 1, D, H, W)
        
        return self.backbone(fused)


class FusionAblation:
    """
    Ablation study comparing different fusion strategies:
    - Mean fusion: Simple average of 4 modalities
    - Static weighted fusion: Fixed weights [0.15, 0.35, 0.25, 0.25]
    - Learnable fusion: Learned weights during training
    """
    def __init__(self, config: TrainConfig, device: torch.device):
        self.config = config
        self.device = device
        self.fusion_types = ['mean', 'static_weighted', 'learnable']
        self.results = {}
    
    def create_model(self, fusion_type: str) -> nn.Module:
        """
        All fusion types return a model that ALWAYS expects 4-channel input.
        The fusion happens INSIDE a wrapper module.
        The backbone always receives 1 channel.
        """
        # 1-channel backbone
        backbone = ResUNet3D(
            in_channels=1,
            num_classes=self.config.num_classes,
            base_filters=self.config.base_filters,
            depth=self.config.depth,
            use_instance_norm=self.config.use_instance_norm,
            dropout_rate_deep=self.config.dropout_rate_deep,
            dropout_rate_shallow=self.config.dropout_rate_shallow
        )
    
        if fusion_type == "mean":
            model = MeanFusionResUNet3D(backbone)
    
        elif fusion_type == "static_weighted":
            model = StaticWeightedFusionResUNet3D(
                num_classes=self.config.num_classes,
                base_filters=self.config.base_filters,
                depth=self.config.depth,
                use_instance_norm=self.config.use_instance_norm,
                dropout_rate_deep=self.config.dropout_rate_deep,
                dropout_rate_shallow=self.config.dropout_rate_shallow
            )
    
        elif fusion_type == "learnable":
            model = LearnableFusionResUNet3D(
                num_classes=self.config.num_classes,
                base_filters=self.config.base_filters,
                depth=self.config.depth,
                use_instance_norm=self.config.use_instance_norm,
                dropout_rate_deep=self.config.dropout_rate_deep,
                dropout_rate_shallow=self.config.dropout_rate_shallow
            )
    
        else:
            raise ValueError(f"Unknown fusion type: {fusion_type}")
    
        model = model.to(self.device)
        
        # Wrap model with DataParallel to use multiple GPUs
        if torch.cuda.device_count() > 1:
            model = nn.DataParallel(model)
        
        return model
    
    def train_and_evaluate(self, fusion_type: str, train_loader: DataLoader, val_loader: DataLoader, epochs: int = 10) -> Dict[str, any]:
        """
        Quick training for ablation (reduced epochs)
        
        Args:
            fusion_type: Type of fusion to use
            train_loader: Training data loader
            val_loader: Validation data loader
            epochs: Number of epochs (reduced for ablation)
            
        Returns:
            Dictionary with history and final metrics
        """
        print(f"\n{'='*60}")
        print(f"Training with {fusion_type.upper()} fusion")
        print(f"{'='*60}")
        
        # Create model
        model = self.create_model(fusion_type)
        print(f"Model parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
        
        # Check if we need 4-channel input
        needs_4ch = fusion_type in ['static_weighted', 'learnable']
        
        # Training components
        optimizer = torch.optim.AdamW(model.parameters(), lr=self.config.learning_rate, weight_decay=1e-5)
        scaler = GradScaler()
        grad_tracker = GradientNormTracker()
        ema_loss = EMALoss(alpha=0.1)
        
        history = {'train_loss': [], 'val_dice': [], 'fusion_weights': []}
        
        for epoch in range(epochs):
            # Training
            model.train()
            total_loss = 0.0
            batch_count = 0
            
            for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
                images = batch['image'].to(self.device)
                labels = batch['label'].to(self.device)
                
                # Handle fusion type
                if fusion_type == 'mean' and images.shape[1] == 4:
                    images = images.mean(dim=1, keepdim=True)
                
                optimizer.zero_grad()
                
                with autocast():
                    outputs = model(images)
                    loss = combined_dice_focal_loss(
                        outputs, labels,
                        alpha=self.config.focal_alpha,
                        gamma=self.config.focal_gamma,
                        dice_weight=self.config.dice_weight,
                        focal_weight=self.config.focal_weight,
                        num_classes=self.config.num_classes
                    )
                
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=self.config.gradient_clip)
                scaler.step(optimizer)
                scaler.update()
                
                total_loss += loss.item()
                batch_count += 1
            
            avg_loss = total_loss / batch_count if batch_count > 0 else 0
            history['train_loss'].append(avg_loss)
            
            # Validation
            model.eval()
            val_metrics = eval_model(
                model=model,
                loader=val_loader,
                device=self.device,
                num_classes=self.config.num_classes,
                num_batches=5,
                use_center_crop=False
            )
            history['val_dice'].append(val_metrics['macro_tumor_dice'])
            
            # Track fusion weights if learnable
            if fusion_type == 'learnable' and hasattr(model, 'get_fusion_weights'):
                weights = model.get_fusion_weights().detach().cpu().numpy().tolist()
                history['fusion_weights'].append(weights)
            
            print(f"Epoch {epoch+1}: Loss={avg_loss:.4f}, Dice={val_metrics['macro_tumor_dice']:.4f}")
        
        # Final evaluation
        final_metrics = eval_model(
            model=model,
            loader=val_loader,
            device=self.device,
            num_classes=self.config.num_classes,
            num_batches=10,
            use_center_crop=False
        )
        
        return {
            'fusion_type': fusion_type,
            'history': history,
            'final_metrics': final_metrics
        }
    
    def run_full_ablation(self, train_loader: DataLoader, val_loader: DataLoader,
                          epochs: int = 10) -> Dict[str, any]:
        """
        Run ablation for all fusion types
        
        Args:
            train_loader: Training data loader (must return 4-channel images)
            val_loader: Validation data loader
            epochs: Number of epochs per fusion type
            
        Returns:
            Dictionary with results for each fusion type
        """
        print("\n" + "="*70)
        print("FUSION ABLATION STUDY")
        print("="*70)
        
        for fusion_type in self.fusion_types:
            result = self.train_and_evaluate(fusion_type, train_loader, val_loader, epochs)
            self.results[fusion_type] = result
        
        return self.results
    
    def plot_comparison(self, output_dir: str = None, show_plot: bool = True) -> None:
        """
        Plot comparison of fusion strategies
        
        Args:
            output_dir: Directory to save plots
            show_plot: Whether to display plots
        """
        if not self.results:
            print("No results to plot. Run ablation first.")
            return
        
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        fig.suptitle('Fusion Strategy Ablation Study', fontsize=14, fontweight='bold')
        
        # Plot 1: Final Macro Tumor Dice comparison
        ax1 = axes[0]
        fusion_types = list(self.results.keys())
        dice_scores = [self.results[ft]['final_metrics']['macro_tumor_dice'] for ft in fusion_types]
        
        bars = ax1.bar(fusion_types, dice_scores, color=['#3498db', '#e74c3c', '#2ecc71'])
        ax1.set_ylabel('Macro Tumor Dice')
        ax1.set_title('Final Performance Comparison')
        ax1.set_ylim([0, 1])
        for bar, score in zip(bars, dice_scores):
            ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{score:.3f}', ha='center', va='bottom', fontsize=10)
        
        # Plot 2: Training curves
        ax2 = axes[1]
        colors = {'mean': '#3498db', 'static_weighted': '#e74c3c', 'learnable': '#2ecc71'}
        for fusion_type, result in self.results.items():
            epochs = range(1, len(result['history']['val_dice']) + 1)
            ax2.plot(epochs, result['history']['val_dice'], label=fusion_type,
                    color=colors.get(fusion_type, 'gray'), linewidth=2)
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Validation Dice')
        ax2.set_title('Training Progress')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # Plot 3: Fusion weights evolution (for learnable)
        ax3 = axes[2]
        if 'learnable' in self.results and self.results['learnable']['history']['fusion_weights']:
            weights_history = self.results['learnable']['history']['fusion_weights']
            modality_names = ['T1', 'T1ce', 'T2', 'FLAIR']
            epochs = range(1, len(weights_history) + 1)
            
            for i, name in enumerate(modality_names):
                weights = [w[i] for w in weights_history]
                ax3.plot(epochs, weights, label=name, linewidth=2)
            
            ax3.set_xlabel('Epoch')
            ax3.set_ylabel('Fusion Weight')
            ax3.set_title('Learnable Fusion Weights Evolution')
            ax3.legend()
            ax3.grid(True, alpha=0.3)
        else:
            ax3.text(0.5, 0.5, 'No learnable fusion\nweights available',
                    ha='center', va='center', transform=ax3.transAxes)
            ax3.set_title('Learnable Fusion Weights')
        
        plt.tight_layout()
        
        if output_dir:
            os.makedirs(output_dir, exist_ok=True)
            save_path = os.path.join(output_dir, 'fusion_ablation.png')
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"Saved: {save_path}")
        
        if show_plot:
            plt.show()
        else:
            plt.close()
    
    def print_summary(self) -> None:
        """Print summary table of results"""
        if not self.results:
            print("No results available. Run ablation first.")
            return
        
        print("\n" + "="*70)
        print("FUSION ABLATION SUMMARY")
        print("="*70)
        print(f"\n{'Fusion Type':<20} {'Macro Dice':>12} {'WT':>8} {'TC':>8} {'ET':>8}")
        print("-"*58)
        
        for fusion_type, result in self.results.items():
            metrics = result['final_metrics']
            print(f"{fusion_type:<20} {metrics['macro_tumor_dice']:>12.4f} "
                  f"{metrics['wt_tc_et']['WT']:>8.4f} "
                  f"{metrics['wt_tc_et']['TC']:>8.4f} "
                  f"{metrics['wt_tc_et']['ET']:>8.4f}")
        
        # Determine best
        best_fusion = max(self.results.keys(),
                         key=lambda x: self.results[x]['final_metrics']['macro_tumor_dice'])
        print("-"*58)
        print(f"Best: {best_fusion} with Dice = "
              f"{self.results[best_fusion]['final_metrics']['macro_tumor_dice']:.4f}")


print("FusionAblation consumes the real train/val loaders (4-channel inputs). Invoke run_full_ablation(train_loader, val_loader, epochs) after Phase 3 data prep if you want comparative results.")


### Cell 26: Loss Function Ablation Study(Nihal)


In [ ]:
# Cell 26: Loss Function Ablation Study
class MeanFusionModel(nn.Module):
    """Wraps any backbone to accept 4-channel MRI and fuse them into 1 channel."""
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, x):
        # x: (B, 4, D, H, W)
        x = x.mean(dim=1, keepdim=True)  # → (B, 1, ...)
        return self.backbone(x)
class LossAblation:
    """
    Ablation study comparing different loss functions:
    - Dice only: Multi-class soft Dice loss
    - Focal only: Focal loss with per-class alpha weights
    - Dice+Focal: Combined loss (default in main training)
    - Cross Entropy: Standard CE loss (baseline)
    
    This study demonstrates:
    1. Class collapse with Dice-only loss
    2. Improved minority class handling with Focal
    3. Best balance with Dice+Focal
    """
    def __init__(self, config: TrainConfig, device: torch.device):
        self.config = config
        self.device = device
        self.loss_types = ['dice_only', 'focal_only', 'dice_focal', 'cross_entropy']
        self.results = {}
    
    def get_loss_fn(self, loss_type: str):
        """
        Get loss function by type
        
        Args:
            loss_type: One of ['dice_only', 'focal_only', 'dice_focal', 'cross_entropy']
            
        Returns:
            Loss function callable
        """
        if loss_type == 'dice_only':
            def loss_fn(pred, target):
                return dice_loss(pred, target, num_classes=self.config.num_classes)
            return loss_fn
        
        elif loss_type == 'focal_only':
            def loss_fn(pred, target):
                return focal_loss(pred, target, alpha=self.config.focal_alpha,
                                  gamma=self.config.focal_gamma)
            return loss_fn
        
        elif loss_type == 'dice_focal':
            def loss_fn(pred, target):
                return combined_dice_focal_loss(
                    pred, target,
                    alpha=self.config.focal_alpha,
                    gamma=self.config.focal_gamma,
                    dice_weight=self.config.dice_weight,
                    focal_weight=self.config.focal_weight,
                    num_classes=self.config.num_classes
                )
            return loss_fn
        
        elif loss_type == 'cross_entropy':
            def loss_fn(pred, target):
                # Standard cross entropy
                pred_flat = pred.permute(0, 2, 3, 4, 1).contiguous().view(-1, self.config.num_classes)
                target_flat = target.view(-1).long()
                return F.cross_entropy(pred_flat, target_flat)
            return loss_fn
        
        else:
            raise ValueError(f"Unknown loss type: {loss_type}")
    
    def train_and_evaluate(self, loss_type: str, train_loader: DataLoader,
                           val_loader: DataLoader, epochs: int = 10) -> Dict[str, any]:
        """
        Quick training with specified loss function
        
        Tracks per-class Dice at each epoch to detect class collapse
        
        Args:
            loss_type: Type of loss function
            train_loader: Training data loader
            val_loader: Validation data loader
            epochs: Number of epochs
            
        Returns:
            Dictionary with history and final metrics
        """
        print(f"\n{'='*60}")
        print(f"Training with {loss_type.upper()} loss")
        print(f"{'='*60}")
        
        # Create model
        # Backbone is 1-channel
        backbone = ResUNet3D(
            in_channels=1,
            num_classes=self.config.num_classes,
            base_filters=self.config.base_filters,
            depth=self.config.depth,
            use_instance_norm=self.config.use_instance_norm,
            dropout_rate_deep=self.config.dropout_rate_deep,
            dropout_rate_shallow=self.config.dropout_rate_shallow
        )

        # Wrap backbone with 4-channel→1-channel fusion
        model = MeanFusionModel(backbone).to(self.device)
        
        # Wrap model with DataParallel to use multiple GPUs
        if torch.cuda.device_count() > 1:
            model = nn.DataParallel(model)
        
        # Get loss function
        loss_fn = self.get_loss_fn(loss_type)
        
        # Training components
        optimizer = torch.optim.AdamW(model.parameters(), lr=self.config.learning_rate, weight_decay=1e-5)
        scaler = GradScaler()
        
        history = {
            'train_loss': [],
            'val_loss': [],
            'per_class_dice': [],  # Track each epoch for collapse detection
            'macro_tumor_dice': []
        }
        
        for epoch in range(epochs):
            # Training
            model.train()
            total_loss = 0.0
            batch_count = 0
            
            for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
                images = batch['image'].to(self.device)
                labels = batch['label'].to(self.device)
                
                # Handle 4-channel input
                if images.shape[1] == 4:
                    images = images.mean(dim=1, keepdim=True)
                
                optimizer.zero_grad()
                
                with autocast():
                    outputs = model(images)
                    loss = loss_fn(outputs, labels)
                
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=self.config.gradient_clip)
                scaler.step(optimizer)
                scaler.update()
                
                total_loss += loss.item()
                batch_count += 1
            
            avg_loss = total_loss / batch_count if batch_count > 0 else 0
            history['train_loss'].append(avg_loss)
            
            # Validation with per-class Dice tracking
            model.eval()
            val_metrics = eval_model(
                model=model,
                loader=val_loader,
                device=self.device,
                num_classes=self.config.num_classes,
                num_batches=5,
                use_center_crop=False
            )
            
            history['per_class_dice'].append(val_metrics['per_class_dice'])
            history['macro_tumor_dice'].append(val_metrics['macro_tumor_dice'])
            
            # Compute validation loss
            val_loss = 0.0
            val_count = 0
            with torch.no_grad():
                for i, batch in enumerate(val_loader):
                    if i >= 5:
                        break
                    images = batch['image'].to(self.device)
                    labels = batch['label'].to(self.device)
                    if images.shape[1] == 4:
                        images = images.mean(dim=1, keepdims=True)
                    with autocast():
                        outputs = model(images)
                        loss = loss_fn(outputs, labels)
                    val_loss += loss.item()
                    val_count += 1
            history['val_loss'].append(val_loss / val_count if val_count > 0 else 0)
            
            # Print progress with per-class Dice
            print(f"Epoch {epoch+1}: Loss={avg_loss:.4f}, Dice={val_metrics['macro_tumor_dice']:.4f}")
            print(f"  Per-class: ", end="")
            for c, score in val_metrics['per_class_dice'].items():
                print(f"C{c}={score:.3f} ", end="")
            print()
            
            # Check for class collapse (any tumor class Dice < 0.01)
            tumor_dices = [val_metrics['per_class_dice'].get(c, 0) for c in range(1, 5)]
            if any(d < 0.01 for d in tumor_dices):
                print(f"  WARNING: Potential class collapse detected!")
        
        # Final evaluation
        final_metrics = eval_model(
            model=model,
            loader=val_loader,
            device=self.device,
            num_classes=self.config.num_classes,
            num_batches=10,
            use_center_crop=False
        )
        
        return {
            'loss_type': loss_type,
            'history': history,
            'final_metrics': final_metrics
        }
    
    def run_full_ablation(self, train_loader: DataLoader, val_loader: DataLoader,
                          epochs: int = 10) -> Dict[str, any]:
        """
        Run ablation for all loss types
        
        Args:
            train_loader: Training data loader
            val_loader: Validation data loader
            epochs: Number of epochs per loss type
            
        Returns:
            Dictionary with results for each loss type
        """
        print("\n" + "="*70)
        print("LOSS FUNCTION ABLATION STUDY")
        print("="*70)
        
        for loss_type in self.loss_types:
            result = self.train_and_evaluate(loss_type, train_loader, val_loader, epochs)
            self.results[loss_type] = result
        
        return self.results
    
    def plot_comparison(self, output_dir: str = None, show_plot: bool = True) -> None:
        """
        Plot comparison of loss functions
        
        Shows:
        1. Per-class Dice evolution (to visualize collapse)
        2. Final performance comparison
        3. Training loss curves
        """
        if not self.results:
            print("No results to plot. Run ablation first.")
            return
        
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        fig.suptitle('Loss Function Ablation Study', fontsize=14, fontweight='bold')
        
        class_names = {0: 'BG', 1: 'Necrosis', 2: 'Edema', 3: 'Non-Enh', 4: 'Enhancing'}
        colors = plt.cm.tab10.colors
        loss_colors = {
            'dice_only': '#e74c3c',
            'focal_only': '#3498db',
            'dice_focal': '#2ecc71',
            'cross_entropy': '#9b59b6'
        }
        
        # Plot 1-4: Per-class Dice evolution for each loss type
        for idx, (loss_type, result) in enumerate(self.results.items()):
            if idx >= 4:
                break
            ax = axes[idx // 2, idx % 2]
            
            epochs = range(1, len(result['history']['per_class_dice']) + 1)
            
            for c in range(5):
                class_dices = [d.get(c, 0) for d in result['history']['per_class_dice']]
                ax.plot(epochs, class_dices, label=class_names[c], 
                       color=colors[c], linewidth=1.5)
            
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Dice Score')
            ax.set_title(f'{loss_type.replace("_", " ").title()} Loss')
            ax.legend(loc='lower right', fontsize=7)
            ax.grid(True, alpha=0.3)
            ax.set_ylim([0, 1])
            
            # Highlight collapse if detected
            tumor_final = [result['history']['per_class_dice'][-1].get(c, 0) for c in range(1, 5)]
            if any(d < 0.01 for d in tumor_final):
                ax.annotate('COLLAPSE', xy=(0.5, 0.5), xycoords='axes fraction',
                           ha='center', va='center', fontsize=16, color='red', alpha=0.5)
        
        # Plot 5: Final performance comparison (bar chart)
        ax5 = axes[1, 0]
        loss_types = list(self.results.keys())
        final_dices = [self.results[lt]['final_metrics']['macro_tumor_dice'] for lt in loss_types]
        
        bars = ax5.bar(range(len(loss_types)), final_dices, 
                      color=[loss_colors.get(lt, 'gray') for lt in loss_types])
        ax5.set_xticks(range(len(loss_types)))
        ax5.set_xticklabels([lt.replace('_', '\n') for lt in loss_types], fontsize=8)
        ax5.set_ylabel('Macro Tumor Dice')
        ax5.set_title('Final Performance Comparison')
        ax5.set_ylim([0, 1])
        
        for bar, score in zip(bars, final_dices):
            ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{score:.3f}', ha='center', va='bottom', fontsize=9)
        
        # Plot 6: Training loss curves
        ax6 = axes[1, 1]
        for loss_type, result in self.results.items():
            epochs = range(1, len(result['history']['train_loss']) + 1)
            ax6.plot(epochs, result['history']['train_loss'], label=loss_type,
                    color=loss_colors.get(loss_type, 'gray'), linewidth=2)
        
        ax6.set_xlabel('Epoch')
        ax6.set_ylabel('Training Loss')
        ax6.set_title('Training Loss Curves')
        ax6.legend(fontsize=8)
        ax6.grid(True, alpha=0.3)
        
        # Hide unused subplot
        axes[1, 2].axis('off')
        
        plt.tight_layout()
        
        if output_dir:
            os.makedirs(output_dir, exist_ok=True)
            save_path = os.path.join(output_dir, 'loss_ablation.png')
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"Saved: {save_path}")
        
        if show_plot:
            plt.show()
        else:
            plt.close()
    
    def print_summary(self) -> None:
        """Print summary table with collapse detection"""
        if not self.results:
            print("No results available. Run ablation first.")
            return
        
        print("\n" + "="*70)
        print("LOSS FUNCTION ABLATION SUMMARY")
        print("="*70)
        
        # Main metrics table
        print(f"\n{'Loss Type':<18} {'Macro Dice':>12} {'WT':>8} {'TC':>8} {'ET':>8} {'Collapse':>10}")
        print("-"*70)
        
        for loss_type, result in self.results.items():
            metrics = result['final_metrics']
            
            # Check for collapse
            tumor_dices = [metrics['per_class_dice'].get(c, 0) for c in range(1, 5)]
            collapsed = "YES" if any(d < 0.01 for d in tumor_dices) else "No"
            
            print(f"{loss_type:<18} {metrics['macro_tumor_dice']:>12.4f} "
                  f"{metrics['wt_tc_et']['WT']:>8.4f} "
                  f"{metrics['wt_tc_et']['TC']:>8.4f} "
                  f"{metrics['wt_tc_et']['ET']:>8.4f} "
                  f"{collapsed:>10}")
        
        # Per-class breakdown
        print("\n" + "-"*70)
        print("Per-Class Dice Scores:")
        print(f"{'Loss Type':<18} {'C0 (BG)':>10} {'C1 (Nec)':>10} {'C2 (Ede)':>10} {'C3 (NE)':>10} {'C4 (Enh)':>10}")
        print("-"*70)
        
        for loss_type, result in self.results.items():
            dice = result['final_metrics']['per_class_dice']
            print(f"{loss_type:<18} ", end="")
            for c in range(5):
                print(f"{dice.get(c, 0):>10.4f} ", end="")
            print()
        
        # Best loss function
        best_loss = max(self.results.keys(),
                       key=lambda x: self.results[x]['final_metrics']['macro_tumor_dice'])
        print("-"*70)
        print(f"\nBest: {best_loss} with Macro Dice = "
              f"{self.results[best_loss]['final_metrics']['macro_tumor_dice']:.4f}")
        print("Recommendation: Use Dice+Focal to prevent class collapse.")


print("LossAblation uses the same real train/val loaders defined earlier. Provide them directly when comparing losses; this notebook no longer executes dummy quick tests.")


## Phase 8: Export and Documentation(Nihal)

### Cell 27: Final Export and Documentation(Nihal)


In [ ]:
# Cell 28: Final Export and Documentation
import pickle
from datetime import datetime


def export_model_weights(model: nn.Module, config: TrainConfig, output_dir: str,
                         model_name: str = "ResUNet3D") -> Dict[str, str]:
    """
    Save model in multiple formats
    
    Args:
        model: Trained model
        config: Training configuration
        output_dir: Directory to save models
        model_name: Name for the model files
        
    Returns:
        Dictionary with paths to saved files
    """
    models_dir = os.path.join(output_dir, 'models')
    os.makedirs(models_dir, exist_ok=True)
    
    saved_files = {}
    
    # Save state dict only (smaller, portable)
    state_dict_path = os.path.join(models_dir, f'{model_name}_state_dict.pt')
    torch.save(model.state_dict(), state_dict_path)
    saved_files['state_dict'] = state_dict_path
    print(f"Saved state dict: {state_dict_path}")
    
    # Save full model with config
    full_model_path = os.path.join(models_dir, f'{model_name}_full.pt')
    torch.save({
        'model_state_dict': model.state_dict(),
        'config': {
            'num_classes': config.num_classes,
            'base_filters': config.base_filters,
            'depth': config.depth,
            'use_instance_norm': config.use_instance_norm,
            'dropout_rate_deep': config.dropout_rate_deep,
            'dropout_rate_shallow': config.dropout_rate_shallow,
        },
        'model_name': model_name,
        'timestamp': datetime.now().isoformat()
    }, full_model_path)
    saved_files['full_model'] = full_model_path
    print(f"Saved full model: {full_model_path}")
    
    # Save model config as JSON
    config_path = os.path.join(models_dir, 'model_config.json')
    config_dict = {
        'model_name': model_name,
        'num_classes': config.num_classes,
        'base_filters': config.base_filters,
        'depth': config.depth,
        'use_instance_norm': config.use_instance_norm,
        'dropout_rate_deep': config.dropout_rate_deep,
        'dropout_rate_shallow': config.dropout_rate_shallow,
        'num_parameters': sum(p.numel() for p in model.parameters())
    }
    with open(config_path, 'w') as f:
        json.dump(config_dict, f, indent=2)
    saved_files['config'] = config_path
    print(f"Saved model config: {config_path}")
    
    return saved_files


def export_metrics_csv(test_results: Dict, history: Dict, output_dir: str) -> Dict[str, str]:
    """
    Export metrics to CSV files
    
    Args:
        test_results: Test evaluation results
        history: Training history
        output_dir: Directory to save CSVs
        
    Returns:
        Dictionary with paths to saved files
    """
    metrics_dir = os.path.join(output_dir, 'metrics')
    os.makedirs(metrics_dir, exist_ok=True)
    
    saved_files = {}
    
    # Test metrics CSV
    test_metrics_data = []
    
    # Add primary metric
    test_metrics_data.append({
        'Category': 'Primary',
        'Metric': 'Macro Tumor Dice',
        'Value': test_results.get('macro_tumor_dice', 0)
    })
    
    # Add WT/TC/ET
    if 'wt_tc_et' in test_results:
        for region, score in test_results['wt_tc_et'].items():
            test_metrics_data.append({
                'Category': 'BraTS',
                'Metric': region,
                'Value': score
            })
    
    # Add per-class Dice
    if 'per_class_dice' in test_results:
        class_names = {0: 'Background', 1: 'Necrosis', 2: 'Edema', 3: 'Non-Enhancing', 4: 'Enhancing'}
        for c, score in test_results['per_class_dice'].items():
            test_metrics_data.append({
                'Category': 'Per-Class Dice',
                'Metric': class_names.get(c, f'Class {c}'),
                'Value': score
            })
    
    test_df = pd.DataFrame(test_metrics_data)
    test_csv_path = os.path.join(metrics_dir, 'test_metrics.csv')
    test_df.to_csv(test_csv_path, index=False)
    saved_files['test_metrics'] = test_csv_path
    print(f"Saved test metrics: {test_csv_path}")
    
    # Training history CSV
    if history:
        history_data = []
        num_epochs = len(history.get('train_loss', []))
        
        for epoch in range(num_epochs):
            row = {'epoch': epoch + 1}
            
            if 'train_loss' in history and epoch < len(history['train_loss']):
                row['train_loss'] = history['train_loss'][epoch]
            
            if 'val_loss' in history and epoch < len(history['val_loss']):
                row['val_loss'] = history['val_loss'][epoch]
            
            if 'macro_tumor_dice' in history and epoch < len(history['macro_tumor_dice']):
                row['macro_tumor_dice'] = history['macro_tumor_dice'][epoch]
            
            if 'learning_rates' in history and epoch < len(history['learning_rates']):
                row['learning_rate'] = history['learning_rates'][epoch]
            
            if 'grad_norms' in history and epoch < len(history['grad_norms']):
                row['grad_norm_mean'] = history['grad_norms'][epoch].get('mean', 0)
                row['grad_norm_max'] = history['grad_norms'][epoch].get('max', 0)
            
            history_data.append(row)
        
        history_df = pd.DataFrame(history_data)
        history_csv_path = os.path.join(metrics_dir, 'training_history.csv')
        history_df.to_csv(history_csv_path, index=False)
        saved_files['training_history'] = history_csv_path
        print(f"Saved training history: {history_csv_path}")
    
    return saved_files


def export_metrics_json(test_results: Dict, history: Dict, config: TrainConfig,
                       output_dir: str) -> str:
    """
    Export comprehensive JSON with all results
    
    Args:
        test_results: Test evaluation results
        history: Training history
        config: Training configuration
        output_dir: Directory to save JSON
        
    Returns:
        Path to saved JSON file
    """
    metrics_dir = os.path.join(output_dir, 'metrics')
    os.makedirs(metrics_dir, exist_ok=True)
    
    # Build comprehensive results object
    results = {
        'timestamp': datetime.now().isoformat(),
        'test_results': {
            'macro_tumor_dice': test_results.get('macro_tumor_dice', 0),
            'wt_tc_et': test_results.get('wt_tc_et', {}),
            'per_class_dice': {str(k): v for k, v in test_results.get('per_class_dice', {}).items()},
            'per_class_iou': {str(k): v for k, v in test_results.get('per_class_iou', {}).items()}
        },
        'training_info': {
            'total_epochs': len(history.get('train_loss', [])) if history else 0,
            'best_epoch': history.get('best_epoch', None) if history else None,
            'best_metric': history.get('best_metric', None) if history else None,
            'final_train_loss': history['train_loss'][-1] if history and history.get('train_loss') else None,
            'final_val_loss': history['val_loss'][-1] if history and history.get('val_loss') else None
        },
        'config': {
            'batch_size': config.batch_size,
            'learning_rate': config.learning_rate,
            'epochs': config.epochs,
            'patch_size': list(config.patch_size),
            'base_filters': config.base_filters,
            'depth': config.depth,
            'num_classes': config.num_classes,
            'use_instance_norm': config.use_instance_norm,
            'dropout_rate_deep': config.dropout_rate_deep,
            'dropout_rate_shallow': config.dropout_rate_shallow,
            'dice_weight': config.dice_weight,
            'focal_weight': config.focal_weight,
            'gradient_clip': config.gradient_clip,
            'gradient_accumulation_steps': config.gradient_accumulation_steps,
            'use_amp': True
        },
        'literature_comparison': {
            'Standard U-Net': {'range': [0.61, 0.889]},
            'Multimodal Fusion U-Net': {'range': [0.75, 0.75]},
            'Res-UNet': {'range': [0.86, 0.9062]},
            'Swin-UNet-EPA': {'range': [0.915, 0.915]}
        }
    }
    
    json_path = os.path.join(metrics_dir, 'results.json')
    with open(json_path, 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"Saved comprehensive results: {json_path}")
    return json_path


def export_training_history(history: Dict, output_dir: str) -> Dict[str, str]:
    """
    Export full training history
    
    Args:
        history: Training history dictionary
        output_dir: Directory to save history
        
    Returns:
        Dictionary with paths to saved files
    """
    history_dir = os.path.join(output_dir, 'history')
    os.makedirs(history_dir, exist_ok=True)
    
    saved_files = {}
    
    if not history:
        print("No history to export.")
        return saved_files
    
    # Save as pickle (full object)
    pkl_path = os.path.join(history_dir, 'history.pkl')
    with open(pkl_path, 'wb') as f:
        pickle.dump(history, f)
    saved_files['pickle'] = pkl_path
    print(f"Saved history pickle: {pkl_path}")
    
    return saved_files


def export_config(config: TrainConfig, output_dir: str) -> Dict[str, str]:
    """
    Export configuration to JSON and text
    
    Args:
        config: Training configuration
        output_dir: Directory to save config
        
    Returns:
        Dictionary with paths to saved files
    """
    reports_dir = os.path.join(output_dir, 'reports')
    os.makedirs(reports_dir, exist_ok=True)
    
    saved_files = {}
    
    # JSON config
    config_dict = {
        'data_root': config.data_root,
        'output_dir': config.output_dir,
        'checkpoint_dir': config.checkpoint_dir,
        'batch_size': config.batch_size,
        'learning_rate': config.learning_rate,
        'epochs': config.epochs,
        'patch_size': list(config.patch_size),
        'num_workers': config.num_workers,
        'base_filters': config.base_filters,
        'depth': config.depth,
        'num_classes': config.num_classes,
        'use_instance_norm': config.use_instance_norm,
        'dropout_rate_deep': config.dropout_rate_deep,
        'dropout_rate_shallow': config.dropout_rate_shallow,
        'dropout_enabled': config.dropout_enabled,
        'optimizer_type': config.optimizer_type,
        'scheduler_type': config.scheduler_type,
        'gradient_clip': config.gradient_clip,
        'gradient_accumulation_steps': config.gradient_accumulation_steps,
        'dice_weight': config.dice_weight,
        'focal_weight': config.focal_weight,
        'focal_alpha': config.focal_alpha,
        'focal_gamma': config.focal_gamma,
        'tumor_centric_ratio': config.tumor_centric_ratio,
        'class_aware_sampling': config.class_aware_sampling,
        'minority_class_ratio': config.minority_class_ratio,
        'use_amp': True,
        'eval_batches': config.eval_batches,
        'use_center_crop_eval': config.use_center_crop_eval
    }
    
    json_path = os.path.join(reports_dir, 'config.json')
    with open(json_path, 'w') as f:
        json.dump(config_dict, f, indent=2)
    saved_files['json'] = json_path
    print(f"Saved config JSON: {json_path}")
    
    # Text config (human-readable)
    txt_path = os.path.join(reports_dir, 'config.txt')
    with open(txt_path, 'w') as f:
        f.write("=" * 60 + "\n")
        f.write("TRAINING CONFIGURATION\n")
        f.write("=" * 60 + "\n\n")
        
        f.write("Data Settings:\n")
        f.write(f"  Data root: {config.data_root}\n")
        f.write(f"  Batch size: {config.batch_size}\n")
        f.write(f"  Patch size: {config.patch_size}\n")
        f.write(f"  Num workers: {config.num_workers}\n\n")
        
        f.write("Model Architecture:\n")
        f.write(f"  Base filters: {config.base_filters}\n")
        f.write(f"  Depth: {config.depth}\n")
        f.write(f"  Num classes: {config.num_classes}\n")
        f.write(f"  Instance Norm: {config.use_instance_norm}\n")
        f.write(f"  Dropout (deep/shallow): {config.dropout_rate_deep}/{config.dropout_rate_shallow}\n\n")
        
        f.write("Training Settings:\n")
        f.write(f"  Optimizer: {config.optimizer_type}\n")
        f.write(f"  Learning rate: {config.learning_rate}\n")
        f.write(f"  Scheduler: {config.scheduler_type}\n")
        f.write(f"  Gradient clip: {config.gradient_clip}\n")
        f.write(f"  Gradient accumulation: {config.gradient_accumulation_steps}\n")
        f.write("  AMP: True\n\n")
        
        f.write("Loss Settings:\n")
        f.write(f"  Dice weight: {config.dice_weight}\n")
        f.write(f"  Focal weight: {config.focal_weight}\n")
        f.write(f"  Focal gamma: {config.focal_gamma}\n")
        f.write(f"  Focal alpha: {config.focal_alpha}\n\n")
        
        f.write("Sampling Settings:\n")
        f.write(f"  Tumor-centric ratio: {config.tumor_centric_ratio}\n")
        f.write(f"  Class-aware sampling: {config.class_aware_sampling}\n")
        f.write(f"  Minority class ratio: {config.minority_class_ratio}\n")
    
    saved_files['txt'] = txt_path
    print(f"Saved config TXT: {txt_path}")
    
    return saved_files


def generate_final_report(test_results: Dict, history: Dict, config: TrainConfig,
                         output_dir: str, model_name: str = "ResUNet3D",
                         ablation_results: Dict = None) -> str:
    """
    Generate comprehensive markdown report
    
    Args:
        test_results: Test evaluation results
        history: Training history
        config: Training configuration
        output_dir: Directory to save report
        model_name: Name of the model
        ablation_results: Optional ablation study results
        
    Returns:
        Path to saved report
    """
    reports_dir = os.path.join(output_dir, 'reports')
    os.makedirs(reports_dir, exist_ok=True)
    
    report_lines = []
    
    # Header
    report_lines.append("# Brain Tumor Segmentation - Final Report")
    report_lines.append("")
    report_lines.append(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    report_lines.append(f"**Model:** {model_name}")
    report_lines.append("")
    
    # Executive Summary
    report_lines.append("## Executive Summary")
    report_lines.append("")
    
    macro_dice = test_results.get('macro_tumor_dice', 0)
    total_epochs = len(history.get('train_loss', [])) if history else 0
    best_epoch = history.get('best_epoch', 'N/A') if history else 'N/A'
    
    report_lines.append(f"- **Primary Metric (Macro Tumor Dice):** {macro_dice*100:.2f}%")
    report_lines.append(f"- **Training Epochs:** {total_epochs}")
    report_lines.append(f"- **Best Epoch:** {best_epoch}")
    report_lines.append("")
    
    # Key Results
    report_lines.append("## Key Results")
    report_lines.append("")
    report_lines.append("| Metric | Score |")
    report_lines.append("|--------|-------|")
    report_lines.append(f"| Macro Tumor Dice | {macro_dice*100:.2f}% |")
    
    if 'wt_tc_et' in test_results:
        report_lines.append(f"| Whole Tumor (WT) | {test_results['wt_tc_et']['WT']*100:.2f}% |")
        report_lines.append(f"| Tumor Core (TC) | {test_results['wt_tc_et']['TC']*100:.2f}% |")
        report_lines.append(f"| Enhancing Tumor (ET) | {test_results['wt_tc_et']['ET']*100:.2f}% |")
    
    report_lines.append("")
    
    # Per-Class Performance
    report_lines.append("## Per-Class Performance")
    report_lines.append("")
    report_lines.append("| Class | Dice Score |")
    report_lines.append("|-------|------------|")
    
    class_names = {0: 'Background', 1: 'Necrosis', 2: 'Edema', 3: 'Non-Enhancing', 4: 'Enhancing'}
    if 'per_class_dice' in test_results:
        for c in range(5):
            score = test_results['per_class_dice'].get(c, 0)
            report_lines.append(f"| {class_names[c]} | {score*100:.2f}% |")
    
    report_lines.append("")
    
    # Literature Comparison
    report_lines.append("## Literature Comparison")
    report_lines.append("")
    report_lines.append("| Method | Dice Range | Our Score | Difference |")
    report_lines.append("|--------|-----------|-----------|------------|")
    
    baselines = {
        'Standard U-Net': (0.61, 0.889),
        'Multimodal Fusion U-Net': (0.75, 0.75),
        'Res-UNet': (0.86, 0.9062),
        'Swin-UNet-EPA': (0.915, 0.915)
    }
    
    for method, (low, high) in baselines.items():
        avg = (low + high) / 2
        diff = macro_dice - avg
        diff_str = f"+{diff*100:.1f}%" if diff > 0 else f"{diff*100:.1f}%"
        report_lines.append(f"| {method} | {low*100:.1f}%-{high*100:.1f}% | {macro_dice*100:.2f}% | {diff_str} |")
    
    report_lines.append("")
    
    # Model Architecture
    report_lines.append("## Model Architecture")
    report_lines.append("")
    report_lines.append(f"- **Architecture:** {model_name}")
    report_lines.append(f"- **Base Filters:** {config.base_filters}")
    report_lines.append(f"- **Depth:** {config.depth}")
    report_lines.append(f"- **Normalization:** {'InstanceNorm3d' if config.use_instance_norm else 'BatchNorm3d'}")
    report_lines.append(f"- **Dropout:** Deep={config.dropout_rate_deep}, Shallow={config.dropout_rate_shallow}")
    report_lines.append("")
    
    # Training Configuration
    report_lines.append("## Training Configuration")
    report_lines.append("")
    report_lines.append(f"- **Batch Size:** {config.batch_size}")
    report_lines.append(f"- **Learning Rate:** {config.learning_rate}")
    report_lines.append(f"- **Optimizer:** {config.optimizer_type}")
    report_lines.append(f"- **Loss:** Dice (weight={config.dice_weight}) + Focal (weight={config.focal_weight})")
    report_lines.append(f"- **Gradient Clipping:** {config.gradient_clip}")
    report_lines.append(f"- **Gradient Accumulation:** {config.gradient_accumulation_steps} steps")
    report_lines.append("")
    
    # Ablation Studies (if available)
    if ablation_results:
        report_lines.append("## Ablation Study Summary")
        report_lines.append("")
        
        for study_name, results in ablation_results.items():
            report_lines.append(f"### {study_name}")
            report_lines.append("")
            # Add ablation-specific summaries here
            report_lines.append("See ablation plots in visualizations folder.")
            report_lines.append("")
    
    # Conclusions
    report_lines.append("## Conclusions")
    report_lines.append("")
    report_lines.append("1. **Class Imbalance:** Dice+Focal loss prevents class collapse")
    report_lines.append("2. **Training Stability:** InstanceNorm3d enables stable training with batch_size=1-2")
    report_lines.append("3. **Multimodal Fusion:** Learnable fusion improves performance over mean fusion")
    report_lines.append("4. **Residual Connections:** Improve gradient flow for deeper networks")
    report_lines.append("")
    
    # Future Work
    report_lines.append("## Future Work")
    report_lines.append("")
    report_lines.append("- Extended training on full dataset")
    report_lines.append("- Data augmentation (rotation, scaling, flipping)")
    report_lines.append("- Post-processing with morphological operations")
    report_lines.append("- Ensemble methods")
    report_lines.append("")
    
    # Write report
    report_path = os.path.join(reports_dir, 'final_report.md')
    with open(report_path, 'w') as f:
        f.write('\n'.join(report_lines))
    
    print(f"Saved final report: {report_path}")
    return report_path


def export_final_results(model: nn.Module, history: Dict, test_results: Dict,
                        config: TrainConfig, output_dir: str = './results',
                        model_name: str = "ResUNet3D",
                        ablation_results: Dict = None) -> Dict[str, any]:
    """
    Export all results, models, and documentation
    
    Main export function that calls all sub-functions.
    
    Args:
        model: Trained model
        history: Training history
        test_results: Test evaluation results
        config: Training configuration
        output_dir: Base output directory
        model_name: Name for the model
        ablation_results: Optional ablation study results
        
    Returns:
        Dictionary with all export paths
    """
    print("\n" + "="*70)
    print("EXPORTING FINAL RESULTS")
    print("="*70)
    print(f"Output directory: {output_dir}")
    
    os.makedirs(output_dir, exist_ok=True)
    
    all_exports = {}
    
    # 1. Export model weights
    print("\n[1/5] Exporting model weights...")
    all_exports['model'] = export_model_weights(model, config, output_dir, model_name)
    
    # 2. Export metrics (CSV)
    print("\n[2/5] Exporting metrics to CSV...")
    all_exports['csv'] = export_metrics_csv(test_results, history, output_dir)
    
    # 3. Export metrics (JSON)
    print("\n[3/5] Exporting comprehensive JSON...")
    all_exports['json'] = export_metrics_json(test_results, history, config, output_dir)
    
    # 4. Export training history
    print("\n[4/5] Exporting training history...")
    all_exports['history'] = export_training_history(history, output_dir)
    
    # 5. Export config
    print("\n[5/5] Exporting configuration...")
    all_exports['config'] = export_config(config, output_dir)
    
    # 6. Generate final report
    print("\n[6/6] Generating final report...")
    all_exports['report'] = generate_final_report(
        test_results, history, config, output_dir, model_name, ablation_results
    )
    
    print("\n" + "="*70)
    print("EXPORT COMPLETE")
    print("="*70)
    print(f"\nOutput structure:")
    print(f"  {output_dir}/")
    print(f"  ├── models/")
    print(f"  │   ├── {model_name}_state_dict.pt")
    print(f"  │   ├── {model_name}_full.pt")
    print(f"  │   └── model_config.json")
    print(f"  ├── metrics/")
    print(f"  │   ├── test_metrics.csv")
    print(f"  │   ├── training_history.csv")
    print(f"  │   └── results.json")
    print(f"  ├── history/")
    print(f"  │   └── history.pkl")
    print(f"  └── reports/")
    print(f"      ├── final_report.md")
    print(f"      ├── config.json")
    print(f"      └── config.txt")
    
    return all_exports


print("Export helpers now assume real training artifacts. After completing training/evaluation, call export_final_results(model, history, test_results, config, output_dir) to persist checkpoints, metrics, and reports.")

print("\n" + "="*70)
print("PHASE 7-8 IMPLEMENTATION COMPLETE")
print("="*70)
print("""
Phase 7: Ablation Studies
- Cell 25: Fusion ablation (mean, static weighted, learnable)
- Cell 26: Loss function ablation (Dice, Focal, Dice+Focal, CE)
- Cell 27: Architecture ablation (Plain U-Net, Swin-UNet, Res-UNet variants)

Phase 8: Export and Documentation
- Cell 28: Final export (models, metrics, history, config, report)

To run ablation studies:
1. fusion_ablation = FusionAblation(config, device)
2. fusion_ablation.run_full_ablation(train_loader, val_loader, epochs=10)
3. fusion_ablation.plot_comparison(output_dir='./outputs')

To export final results:
1. export_final_results(model, history, test_results, config, './results')

All phases complete! Ready for training on real data.
""")



## Phase 9: Automated Execution Pipelines(Nihal)
These closing cells orchestrate the entire project in two passes: first run the training pipeline to produce checkpoints/plots, then (optionally after restarting the kernel and re-running Phases 1–8) run the evaluation pipeline to reload the saved model, execute all ablation studies, and export every artifact.

### Cell 28: Training Execution & Plotting(Nihal)
Run this cell after completing Phases 1–8 to train on the real BraTS loaders, save checkpoints/history, and emit all training plots in one go. Toggle the flag only when you are ready for a full training run.


In [ ]:

import pickle

TRAINING_PHASE_ENABLED = False  # Flip to True to run full training/plotting
TRAINING_HISTORY_PATH = os.path.join(config.output_dir, 'history', 'phase9_training_history.pkl')
BEST_CHECKPOINT_PATH = os.path.join(config.checkpoint_dir, 'best_model.pt')

if TRAINING_PHASE_ENABLED:
    if 'train_loader' not in locals() or 'val_loader' not in locals():
        raise RuntimeError("train_loader/val_loader not found. Re-run the data loading cells in Phase 3 before launching training.")
    if len(train_loader) == 0 or len(val_loader) == 0:
        raise RuntimeError("train_loader/val_loader are empty. Verify dataset discovery and patch sampling before training.")

    print("[Phase 9] Starting full training run...")
    phase9_model = create_model(config, device)
    training_history = train_model(
        model=phase9_model,
        train_loader=train_loader,
        val_loader=val_loader,
        config=config,
        device=device
    )

    # Persist training history for later evaluation-only runs
    os.makedirs(os.path.dirname(TRAINING_HISTORY_PATH), exist_ok=True)
    with open(TRAINING_HISTORY_PATH, 'wb') as f:
        pickle.dump(training_history, f)
    export_training_history(training_history, config.output_dir)

    # Generate training plots (saved to output_dir)
    plot_training_curves(training_history, output_dir=config.output_dir, show_plot=False)
    plot_class_comparison(training_history, output_dir=config.output_dir, show_plot=False)

    print(f"Training history saved to: {TRAINING_HISTORY_PATH}")
    print(f"Best checkpoint path: {BEST_CHECKPOINT_PATH}")
else:
    print("Training phase disabled. Set TRAINING_PHASE_ENABLED=True to run the full pipeline training pass.")


### Cell 29: Evaluation, Ablations, and Final Export(Nihal)
Run this cell after the training cell has produced `best_model.pt` and `phase9_training_history.pkl`. It reloads the saved checkpoint, evaluates on the validation loader, executes **all** ablation studies (fusion, loss, architecture), and exports every artifact.


In [ ]:
EVALUATION_PHASE_ENABLED = True  # Flip to True for evaluation/ablation/export
ABLA_TRAIN_HISTORY_PATH = TRAINING_HISTORY_PATH  # Reuse Phase 9 history path
ABLA_EPOCHS = max(2, min(5, config.epochs // 8))
VISUALIZATION_DIR = os.path.join(config.output_dir, 'visualizations')

if EVALUATION_PHASE_ENABLED:
    if 'train_loader' not in locals() or 'val_loader' not in locals():
        raise RuntimeError("train_loader/val_loader not found. Re-run Phase 2/3 data cells before executing evaluation.")
    if 'sampler' not in locals() or 'train_cases' not in locals() or 'val_cases' not in locals():
        raise RuntimeError("sampler/train_cases/val_cases not found. Ensure the data discovery cells executed successfully.")
    if not os.path.exists(BEST_CHECKPOINT_PATH):
        raise FileNotFoundError(f"Best-model checkpoint not found at {BEST_CHECKPOINT_PATH}. Run the training cell first.")

    # Reload training history if necessary
    if 'training_history' in locals():
        evaluation_history = training_history
    elif os.path.exists(ABLA_TRAIN_HISTORY_PATH):
        with open(ABLA_TRAIN_HISTORY_PATH, 'rb') as f:
            evaluation_history = pickle.load(f)
    else:
        evaluation_history = {}
        print("Warning: Training history not found; proceeding without history metadata.")

    print("[Phase 9] Loading best checkpoint and running validation evaluation...")
    evaluation_model, checkpoint_info = load_best_model(
        checkpoint_path=BEST_CHECKPOINT_PATH,
        config=config,
        device=device
    )

    validation_eval = run_full_evaluation_pipeline(
        checkpoint_path=BEST_CHECKPOINT_PATH,
        test_loader=val_loader,
        config=config,
        device=device,
        history=evaluation_history,
        output_dir=config.output_dir
    )
    test_results = validation_eval['test_results']

    # Extra qualitative overlays (saved silently)
    os.makedirs(VISUALIZATION_DIR, exist_ok=True)
    visualize_predictions(
        model=evaluation_model,
        loader=val_loader,
        device=device,
        num_samples=3,
        output_dir=VISUALIZATION_DIR,
        show_plot=False
    )

    print("[Phase 9] Building 4-channel dataloaders for ablation studies...")
    ablation_train_dataset = BRATSPatchDataset(
        cases=train_cases,
        sampler=sampler,
        fusion_mode='learnable',
        num_patches_per_case=4
    )
    ablation_val_dataset = BRATSPatchDataset(
        cases=val_cases,
        sampler=sampler,
        fusion_mode='learnable',
        num_patches_per_case=2
    )
    ablation_train_loader, ablation_val_loader, _ = create_dataloaders(
        ablation_train_dataset,
        ablation_val_dataset,
        None,
        config
    )

    ablation_results = {}

    print("[Phase 9] Running fusion ablation (mean/static/learnable)...")
    fusion_ablation = FusionAblation(config, device)
    fusion_results = fusion_ablation.run_full_ablation(
        train_loader=ablation_train_loader,
        val_loader=ablation_val_loader,
        epochs=ABLA_EPOCHS
    )
    fusion_ablation.plot_comparison(output_dir=config.output_dir, show_plot=False)
    fusion_ablation.print_summary()
    ablation_results['fusion'] = fusion_results

    print("[Phase 9] Running loss-function ablation (dice/focal/dice+focal/CE)...")
    loss_ablation = LossAblation(config, device)
    loss_results = loss_ablation.run_full_ablation(
        train_loader=ablation_train_loader,
        val_loader=ablation_val_loader,
        epochs=ABLA_EPOCHS
    )
    loss_ablation.plot_comparison(output_dir=config.output_dir, show_plot=False)
    loss_ablation.print_summary()
    ablation_results['loss'] = loss_results

    print("[Phase 9] Exporting all artifacts (models, metrics, history, reports)...")
    export_payload = export_final_results(
        model=evaluation_model,
        history=evaluation_history if evaluation_history else {},
        test_results=test_results,
        config=config,
        output_dir=config.output_dir,
        model_name="LearnableFusionResUNet3D" if config.use_learnable_fusion else "ResUNet3D",
        ablation_results=ablation_results
    )
else:
    print("Evaluation phase disabled. Set EVALUATION_PHASE_ENABLED=True to load the saved checkpoint, run ablations, and export final artifacts.")



In [ ]:
# Download all output folders from Kaggle
import os
from IPython.display import FileLink

# Create zip with both outputs and checkpoints folders
zip_filename = 'kaggle_outputs.zip'

# Remove existing zip if present
if os.path.exists(zip_filename):
    os.remove(zip_filename)

# Zip both folders (outputs and checkpoints)
os.system(f'zip -r {zip_filename} ./outputs ./checkpoints > /dev/null 2>&1')

# Provide download link
FileLink(zip_filename)